In [1]:
%pip install requests pandas

932.86s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import pandas as pd
print ("Ready to start the Earthquake Project!")

Ready to start the Earthquake Project!


In [6]:
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

parameters = {
    "format": "geojson",
    "starttime": "2026-10-01",
    "endtime": "2026-10-08",
    "limit": 1000,
    "orderby": "time"
}

response = requests.get(url, params=parameters, timeout=30)

response.raise_for_status()

data = response.json()

print("API connection successful!")
print("Earthquake records found:", len(data["features"]))
print("Available metadata:", data["metadata"].keys())

API connection successful!
Earthquake records found: 1000
Available metadata: dict_keys(['generated', 'url', 'title', 'status', 'api', 'limit', 'offset'])


In [4]:
# Check what information the API returned
print(data["metadata"])

{'generated': 1792822842000, 'url': 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2026-10-01&endtime=2026-10-08&limit=1000&orderby=time', 'title': 'USGS Earthquakes', 'status': 200, 'api': '2.8.2', 'limit': 1000, 'offset': 1}


In [7]:
   # Get the first earthquake record
first_event = data["features"][0]

# Display its ID
print("Earthquake ID:", first_event["id"])

# Display its properties
print("Earthquake properties:")
print(first_event["properties"])

# Display its coordinates
print("Coordinates:")
print(first_event["geometry"]["coordinates"])

Earthquake ID: av94464641
Earthquake properties:
{'mag': -0.81, 'place': '63 km ENE of Pedro Bay, Alaska', 'time': 1791417239950, 'updated': 1791492032570, 'tz': None, 'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/av94464641', 'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=av94464641&format=geojson', 'felt': None, 'cdi': None, 'mmi': None, 'alert': None, 'status': 'reviewed', 'tsunami': 0, 'sig': 0, 'net': 'av', 'code': '94464641', 'ids': ',av94464641,', 'sources': ',av,', 'types': ',origin,phase-data,', 'nst': 5, 'dmin': 0.02599, 'rms': 0.19, 'gap': 108, 'magType': 'ml', 'type': 'earthquake', 'title': 'M -0.8 - 63 km ENE of Pedro Bay, Alaska'}
Coordinates:
[-153.067, 60.0253333333333, 3.88]


In [9]:
earthquake_records = []

for event in data["features"]:

    properties = event["properties"]
    coordinates = event["geometry"]["coordinates"]

    record = {
        "id": event["id"],
        "time": properties.get("time"),
        "updated": properties.get("updated"),
        "latitude": coordinates[1],
        "longitude": coordinates[0],
        "depth_km": coordinates[2],
        "mag": properties.get("mag"),
        "magType": properties.get("magType"),
        "place": properties.get("place"),
        "status": properties.get("status"),
        "tsunami": properties.get("tsunami"),
        "sig": properties.get("sig"),
        "net": properties.get("net"),
        "nst": properties.get("nst"),
        "dmin": properties.get("dmin"),
        "rms": properties.get("rms"),
        "gap": properties.get("gap"),
        "magError": properties.get("magError"),
        "depthError": properties.get("depthError"),
        "magNst": properties.get("magNst"),
        "locationSource": properties.get("locationSource"),
        "magSource": properties.get("magSource"),
        "types": properties.get("types"),
        "ids": properties.get("ids"),
        "sources": properties.get("sources"),
        "type": properties.get("type")
    }

    earthquake_records.append(record)

print("Records extracted:", len(earthquake_records))

Records extracted: 1000


In [10]:
df = pd.DataFrame(earthquake_records)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (1000, 26)


,id,time,updated,latitude,longitude,...,magSource,types,ids,sources,type
0,av94464641,1791417239950,1791492032570,60.025333,-153.067000,...,None,",origin,phase-data,",",av94464641,",",av,",earthquake
1,ci41344751,1791416322790,1791491424724,33.814667,-117.695333,...,None,",nearby-cities,origin,phase-data,scitech-link,",",ci41344751,",",ci,",earthquake
2,aka2026txdhrc,1791416114022,1792797458040,64.637000,-151.319000,...,None,",origin,phase-data,",",us6000u0va,aka2026txdhrc,",",us,ak,",earthquake
3,ci41344743,1791415406330,1791468024720,33.409333,-116.405500,...,None,",focal-mechanism,nearby-cities,origin,phase-data,",",ci41344743,",",ci,",earthquake
4,nc75446246,1791415370080,1792439888751,37.426500,-121.454000,...,None,",nearby-cities,origin,phase-data,scitech-link,",",nc75446246,",",nc,",earthquake


In [11]:
print("Number of columns:", len(df.columns))
print(df.columns.tolist())

Number of columns: 26
['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag', 'magType', 'place', 'status', 'tsunami', 'sig', 'net', 'nst', 'dmin', 'rms', 'gap', 'magError', 'depthError', 'magNst', 'locationSource', 'magSource', 'types', 'ids', 'sources', 'type']


In [12]:
missing_values = df.isnull().sum()

print(missing_values)

id                   0
time                 0
updated              0
latitude             0
longitude            0
depth_km             0
mag                  0
magType              0
place                0
status               0
tsunami              0
sig                  0
net                  0
nst                  0
dmin                 0
rms                  0
gap                  0
magError          1000
depthError        1000
magNst            1000
locationSource    1000
magSource         1000
types                0
ids                  0
sources              0
type                 0
dtype: int64


In [13]:
df["time"] = pd.to_datetime(
    df["time"], unit="ms", utc=True
)

df["updated"] = pd.to_datetime(
    df["updated"], unit="ms", utc=True
)

df[["id", "time", "updated", "place"]].head()

,id,time,updated,place
0,av94464641,2026-10-07 23:53:59.950000+00:00,2026-10-08 20:40:32.570000+00:00,"63 km ENE of Pedro Bay, Alaska"
1,ci41344751,2026-10-07 23:38:42.790000+00:00,2026-10-08 20:30:24.724000+00:00,"11 km ENE of North Tustin, CA"
2,aka2026txdhrc,2026-10-07 23:35:14.022000+00:00,2026-10-23 23:17:38.040000+00:00,"52 km SW of Manley Hot Springs, Alaska"
3,ci41344743,2026-10-07 23:23:26.330000+00:00,2026-10-08 14:00:24.720000+00:00,"17 km N of Borrego Springs, CA"
4,nc75446246,2026-10-07 23:22:50.080000+00:00,2026-10-19 19:58:08.751000+00:00,"26 km WSW of Westley, CA"


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 26 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   id              1000 non-null   object             
 1   time            1000 non-null   datetime64[ns, UTC]
 2   updated         1000 non-null   datetime64[ns, UTC]
 3   latitude        1000 non-null   float64            
 4   longitude       1000 non-null   float64            
 5   depth_km        1000 non-null   float64            
 6   mag             1000 non-null   float64            
 7   magType         1000 non-null   object             
 8   place           1000 non-null   object             
 9   status          1000 non-null   object             
 10  tsunami         1000 non-null   int64              
 11  sig             1000 non-null   int64              
 12  net             1000 non-null   object             
 13  nst             1000 non-null   in

In [15]:
columns_to_check = [
    "magError",
    "depthError",
    "magNst",
    "locationSource",
    "magSource"
]

print(df[columns_to_check].isnull().sum())

magError          1000
depthError        1000
magNst            1000
locationSource    1000
magSource         1000
dtype: int64


In [16]:
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day
df["day_of_week"] = df["time"].dt.day_name()

df[["time", "year", "month", "day", "day_of_week"]].head()

,time,year,month,day,day_of_week
0,2026-10-07 23:53:59.950000+00:00,2026,10,7,Wednesday
1,2026-10-07 23:38:42.790000+00:00,2026,10,7,Wednesday
2,2026-10-07 23:35:14.022000+00:00,2026,10,7,Wednesday
3,2026-10-07 23:23:26.330000+00:00,2026,10,7,Wednesday
4,2026-10-07 23:22:50.080000+00:00,2026,10,7,Wednesday


In [17]:
print("Duplicate event IDs:", df["id"].duplicated().sum())

Duplicate event IDs: 0


In [18]:
df["depth_category"] = df["depth_km"].apply(
    lambda depth: "Shallow" if depth < 50 else "Deep"
)

df[["place", "depth_km", "depth_category"]].head()

,place,depth_km,depth_category
0,"63 km ENE of Pedro Bay, Alaska",3.88,Shallow
1,"11 km ENE of North Tustin, CA",-0.48,Shallow
2,"52 km SW of Manley Hot Springs, Alaska",14.10,Shallow
3,"17 km N of Borrego Springs, CA",8.05,Shallow
4,"26 km WSW of Westley, CA",4.05,Shallow


In [19]:
df["strong_earthquake"] = df["mag"] >= 6.0

df[["place", "mag", "strong_earthquake"]].head()

,place,mag,strong_earthquake
0,"63 km ENE of Pedro Bay, Alaska",-0.81,False
1,"11 km ENE of North Tustin, CA",0.33,False
2,"52 km SW of Manley Hot Springs, Alaska",2.60,False
3,"17 km N of Borrego Springs, CA",1.09,False
4,"26 km WSW of Westley, CA",1.14,False


In [20]:
print("Earthquakes by depth category:")
print(df["depth_category"].value_counts())

print("\nStrong earthquake flag:")
print(df["strong_earthquake"].value_counts())

Earthquakes by depth category:
depth_category
Shallow    842
Deep       158
Name: count, dtype: int64

Strong earthquake flag:
strong_earthquake
False    1000
Name: count, dtype: int64


In [21]:
df.to_csv("earthquakes_sample_processed.csv", index=False)

print("Processed sample saved successfully!")

Processed sample saved successfully!


In [22]:
from datetime import date

start_date = date(2021, 10, 24)
end_date = date(2026, 10, 25)

print("Start date:", start_date)
print("End date:", end_date)

Start date: 2021-10-24
End date: 2026-10-25


In [23]:
print("Minimum magnitude:", df["mag"].min())
print("Maximum magnitude:", df["mag"].max())
print("\nMagnitude summary:")
print(df["mag"].describe())

Minimum magnitude: -1.29
Maximum magnitude: 5.8

Magnitude summary:
count    1000.000000
mean        1.640780
std         1.234801
min        -1.290000
25%         0.880000
50%         1.500000
75%         2.100000
max         5.800000
Name: mag, dtype: float64


In [24]:
import time
from datetime import date
import requests
import pandas as pd

# Five-year date range
start_date = date(2021, 10, 24)
end_date = date(2026, 10, 25)  # Exclusive upper boundary

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

all_records = []
session = requests.Session()

current_date = start_date

while current_date < end_date:

    # Calculate the first day of the next month
    if current_date.month == 12:
        next_month = date(current_date.year + 1, 1, 1)
    else:
        next_month = date(
            current_date.year,
            current_date.month + 1,
            1
        )

    month_end = min(next_month, end_date)

    offset = 1
    month_count = 0

    while True:

        parameters = {
            "format": "geojson",
            "starttime": current_date.isoformat(),
            "endtime": month_end.isoformat(),
            "limit": 20000,
            "offset": offset
        }

        response = session.get(
            url,
            params=parameters,
            timeout=60
        )
        response.raise_for_status()

        batch = response.json()["features"]

        if not batch:
            break

        for event in batch:
            properties = event["properties"]
            coordinates = (
                event.get("geometry") or {}
            ).get("coordinates") or [None, None, None]

            coordinates = list(coordinates) + [None] * (
                3 - len(coordinates)
            )

            record = {
                "id": event.get("id"),
                "time": properties.get("time"),
                "updated": properties.get("updated"),
                "latitude": coordinates[1],
                "longitude": coordinates[0],
                "depth_km": coordinates[2],
                "mag": properties.get("mag"),
                "magType": properties.get("magType"),
                "place": properties.get("place"),
                "status": properties.get("status"),
                "tsunami": properties.get("tsunami"),
                "sig": properties.get("sig"),
                "net": properties.get("net"),
                "nst": properties.get("nst"),
                "dmin": properties.get("dmin"),
                "rms": properties.get("rms"),
                "gap": properties.get("gap"),
                "magError": properties.get("magError"),
                "depthError": properties.get("depthError"),
                "magNst": properties.get("magNst"),
                "locationSource": properties.get("locationSource"),
                "magSource": properties.get("magSource"),
                "types": properties.get("types"),
                "ids": properties.get("ids"),
                "sources": properties.get("sources"),
                "type": properties.get("type")
            }

            all_records.append(record)

        month_count += len(batch)
        offset += len(batch)

        # A short pause between requests
        time.sleep(0.1)

        if len(batch) < 20000:
            break

    print(
        f"Completed through {month_end}: "
        f"{month_count:,} records"
    )

    current_date = month_end

# Create the complete DataFrame
df = pd.DataFrame(all_records)

# Remove duplicate events if date boundaries overlap
df = df.drop_duplicates(subset="id").reset_index(drop=True)

print("\nDOWNLOAD COMPLETE")
print("Dataset shape:", df.shape)
print("Unique event IDs:", df["id"].nunique())

Completed through 2021-11-01: 3,207 records
Completed through 2021-12-01: 11,583 records
Completed through 2022-01-01: 11,400 records
Completed through 2022-02-01: 12,954 records
Completed through 2022-03-01: 11,768 records
Completed through 2022-04-01: 12,803 records
Completed through 2022-05-01: 12,430 records
Completed through 2022-06-01: 12,167 records
Completed through 2022-07-01: 11,851 records
Completed through 2022-08-01: 12,231 records
Completed through 2022-09-01: 13,019 records
Completed through 2022-10-01: 12,996 records
Completed through 2022-11-01: 13,066 records
Completed through 2022-12-01: 12,647 records
Completed through 2023-01-01: 12,808 records
Completed through 2023-02-01: 12,199 records
Completed through 2023-03-01: 11,754 records
Completed through 2023-04-01: 15,792 records
Completed through 2023-05-01: 13,349 records
Completed through 2023-06-01: 12,823 records
Completed through 2023-07-01: 12,528 records
Completed through 2023-08-01: 12,968 records
Completed t

In [25]:
df.to_csv("earthquakes_5years_raw.csv", index=False)

print("Five-year raw dataset saved!")

Five-year raw dataset saved!


In [26]:
print("Rows and columns:", df.shape)
print("Date range:")

print(
    pd.to_datetime(df["time"], unit="ms", utc=True).min()
)

print(
    pd.to_datetime(df["time"], unit="ms", utc=True).max()
)

Rows and columns: (739785, 26)
Date range:
2021-10-24 00:02:44.820000+00:00
2026-10-24 06:31:36.660000+00:00


In [27]:
# Convert timestamps from milliseconds to datetime
df["time"] = pd.to_datetime(
    df["time"], unit="ms", utc=True
)

df["updated"] = pd.to_datetime(
    df["updated"], unit="ms", utc=True
)

# Check the dataset
print("Dataset shape:", df.shape)

print("\nEarliest earthquake:")
print(df["time"].min())

print("\nLatest earthquake:")
print(df["time"].max())

df[["id", "time", "mag", "depth_km", "place"]].head()

Dataset shape: (739785, 26)

Earliest earthquake:
2021-10-24 00:02:44.820000+00:00

Latest earthquake:
2026-10-24 06:31:36.660000+00:00


,id,time,mag,depth_km,place
0,us7000fqcd,2021-10-31 23:47:49.663000+00:00,4.40,570.45,"235 km E of Levuka, Fiji"
1,nc73647215,2021-10-31 23:42:49.670000+00:00,0.85,1.47,"2km NNW of The Geysers, CA"
2,us7000fqca,2021-10-31 23:39:39.089000+00:00,4.40,496.53,south of the Fiji Islands
3,hv72777307,2021-10-31 23:37:23.860000+00:00,1.52,33.67,"6 km E of P?hala, Hawaii"
4,ci39845863,2021-10-31 23:34:04.820000+00:00,1.46,9.14,"13km NE of Ridgecrest, CA"


In [28]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().sum() / len(df) * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    "Missing_Percentage",
    ascending=False
)

missing_summary

,Missing_Count,Missing_Percentage
magSource,739785,100.00
locationSource,739785,100.00
magNst,739785,100.00
depthError,739785,100.00
magError,739785,100.00
dmin,195858,26.47
nst,158622,21.44
gap,147873,19.99
mag,55,0.01
magType,55,0.01


In [29]:
print(
    "Duplicate event IDs:",
    df["id"].duplicated().sum()
)

print(
    "Total records:",
    len(df)
)

Duplicate event IDs: 0
Total records: 739785


In [30]:
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day
df["day_of_week"] = df["time"].dt.day_name()

df[[
    "time",
    "year",
    "month",
    "day",
    "day_of_week"
]].head()

,time,year,month,day,day_of_week
0,2021-10-31 23:47:49.663000+00:00,2021,10,31,Sunday
1,2021-10-31 23:42:49.670000+00:00,2021,10,31,Sunday
2,2021-10-31 23:39:39.089000+00:00,2021,10,31,Sunday
3,2021-10-31 23:37:23.860000+00:00,2021,10,31,Sunday
4,2021-10-31 23:34:04.820000+00:00,2021,10,31,Sunday


In [31]:
df.to_csv(
    "earthquakes_5years_processed.csv",
    index=False
)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!


In [32]:
important_columns = [
    "mag",
    "magType",
    "depth_km",
    "latitude",
    "longitude",
    "place",
    "dmin",
    "nst",
    "gap",
    "rms"
]

print(df[important_columns].isnull().sum())

mag              55
magType          55
depth_km          0
latitude          0
longitude         0
place             0
dmin         195858
nst          158622
gap          147873
rms              57
dtype: int64


In [33]:
df_analysis = df.dropna(
    subset=["mag", "depth_km", "latitude", "longitude"]
).copy()

print("Original records:", len(df))
print("Records available for analysis:", len(df_analysis))
print(
    "Records excluded from this analysis:",
    len(df) - len(df_analysis)
)

Original records: 739785
Records available for analysis: 739730
Records excluded from this analysis: 55


In [34]:
print("Missing magnitude:", df_analysis["mag"].isnull().sum())
print("Missing depth:", df_analysis["depth_km"].isnull().sum())
print("Missing latitude:", df_analysis["latitude"].isnull().sum())
print("Missing longitude:", df_analysis["longitude"].isnull().sum())

Missing magnitude: 0
Missing depth: 0
Missing latitude: 0
Missing longitude: 0


In [36]:
import re

In [37]:
df[["place"]].head(20)

,place
0,"235 km E of Levuka, Fiji"
1,"2km NNW of The Geysers, CA"
2,south of the Fiji Islands
3,"6 km E of P?hala, Hawaii"
4,"13km NE of Ridgecrest, CA"
5,"22 km NE of Hachinohe, Japan"
6,"4km NNW of Boron, CA"
7,"5 km SSE of P?hala, Hawaii"
8,"5 km S of P?hala, Hawaii"
9,"12 km ENE of Fox, Alaska"


In [38]:
df["location_region"] = df["place"].str.extract(
    r",\s*([^,]+)$"
)

df[["place", "location_region"]].head(20)

,place,location_region
0,"235 km E of Levuka, Fiji",Fiji
1,"2km NNW of The Geysers, CA",CA
2,south of the Fiji Islands,NaN
3,"6 km E of P?hala, Hawaii",Hawaii
4,"13km NE of Ridgecrest, CA",CA
5,"22 km NE of Hachinohe, Japan",Japan
6,"4km NNW of Boron, CA",CA
7,"5 km SSE of P?hala, Hawaii",Hawaii
8,"5 km S of P?hala, Hawaii",Hawaii
9,"12 km ENE of Fox, Alaska",Alaska


In [39]:
df["location_region"].value_counts().head(20)

location_region
Alaska              250125
CA                  195783
Hawaii               37684
Texas                25847
Nevada               22436
Oklahoma             22252
Washington           15869
Puerto Rico          14166
Utah                 12286
Montana              11023
New Mexico           10987
Indonesia             8405
Russia                6252
California            5263
Philippines           4820
Japan                 4714
Idaho                 4689
Wyoming               4090
Papua New Guinea      3395
Oregon                3255
Name: count, dtype: int64

In [40]:
print(
    "Total earthquake records:",
    len(df)
)

print(
    "Records with extracted location text:",
    df["location_region"].notna().sum()
)

print(
    "Records without extracted location text:",
    df["location_region"].isna().sum()
)

Total earthquake records: 739785
Records with extracted location text: 711821
Records without extracted location text: 27964


In [41]:
# Show examples where location extraction succeeded
df[["place", "location_region"]].dropna().sample(
    15, random_state=42
)

,place,location_region
341371,"265 km WSW of Crescent City, California",California
52456,"12km ENE of Ocotillo Wells, CA",CA
377942,"Pagan region, Northern Mariana Islands",Northern Mariana Islands
634156,"204 km S of Kencong, Indonesia",Indonesia
560513,"83 km SSE of Sand Point, Alaska",Alaska
261423,"Fox Islands, Aleutian Islands, Alaska",Alaska
50459,"55 km E of Selawik, Alaska",Alaska
93488,"23 km ESE of Skwentna, Alaska",Alaska
723151,"7 km SW of Pāhala, Hawaii",Hawaii
287398,"15 km NE of Kalama, Washington",Washington


In [42]:
df.loc[
    df["location_region"].isna(),
    ["place"]
].head(15)

,place
2,south of the Fiji Islands
34,southern East Pacific Rise
128,Kermadec Islands region
141,Banda Sea
142,south of Alaska
187,southeast of the Loyalty Islands
229,Kermadec Islands region
252,West Chile Rise
297,northern Alaska
318,north of Ascension Island


In [43]:
import re

def extract_location(place):
    if pd.isna(place):
        return None

    place = place.strip()

    # If a comma exists, take the final segment
    if "," in place:
        return place.rsplit(",", 1)[-1].strip()

    # Otherwise, retain the full location description
    return place

df["location_region"] = df["place"].apply(extract_location)

df[["place", "location_region"]].head(20)

,place,location_region
0,"235 km E of Levuka, Fiji",Fiji
1,"2km NNW of The Geysers, CA",CA
2,south of the Fiji Islands,south of the Fiji Islands
3,"6 km E of P?hala, Hawaii",Hawaii
4,"13km NE of Ridgecrest, CA",CA
5,"22 km NE of Hachinohe, Japan",Japan
6,"4km NNW of Boron, CA",CA
7,"5 km SSE of P?hala, Hawaii",Hawaii
8,"5 km S of P?hala, Hawaii",Hawaii
9,"12 km ENE of Fox, Alaska",Alaska


In [44]:
print("Total records:", len(df))
print(
    "Locations extracted:",
    df["location_region"].notna().sum()
)
print(
    "Locations still missing:",
    df["location_region"].isna().sum()
)

Total records: 739785
Locations extracted: 739785
Locations still missing: 0


In [45]:
df[["place", "location_region"]].sample(
    15, random_state=42
)

,place,location_region
128444,"13km ENE of Ridgecrest, CA",CA
40688,"100 km WNW of La Libertad, Mexico",Mexico
234288,"20km N of San Simeon, CA",CA
517363,"14 km SW of Blyn, Washington",Washington
333252,"76 km SW of Waingapu, Indonesia",Indonesia
652358,"7 km WNW of Cobb, CA",CA
427362,"1 km WNW of Mullan, Idaho",Idaho
283361,"14 km WSW of Toms Place, CA",CA
198218,"2 km SSE of Büyük Dalyan, Turkey",Turkey
464472,"12 km WSW of Hardtner, Kansas",Kansas


In [46]:
# Classify earthquakes by depth
df["depth_category"] = df["depth_km"].apply(
    lambda x: "Shallow" if x < 50 else "Deep"
)

# Flag earthquakes with magnitude 6 or above
df["strong_earthquake"] = df["mag"].ge(6.0)

# Create a separate flag for magnitude 7 or above
df["major_earthquake"] = df["mag"].ge(7.0)

# Create date-based features
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day
df["day_of_week"] = df["time"].dt.day_name()

print("Derived features created successfully!")

Derived features created successfully!


In [47]:
print("Earthquakes by depth:")
print(df["depth_category"].value_counts())

print("\nEarthquakes with magnitude >= 6:")
print(df["strong_earthquake"].value_counts())

print("\nEarthquakes with magnitude >= 7:")
print(df["major_earthquake"].value_counts())

Earthquakes by depth:
depth_category
Shallow    643848
Deep        95937
Name: count, dtype: int64

Earthquakes with magnitude >= 6:
strong_earthquake
False    739120
True        665
Name: count, dtype: int64

Earthquakes with magnitude >= 7:
major_earthquake
False    739713
True         72
Name: count, dtype: int64


In [49]:
df.to_csv(
    "earthquakes_5years_processed.csv",
    index=False
)

print("Processed five-year dataset saved successfully!")
print("Final shape:", df.shape)

Processed five-year dataset saved successfully!
Final shape: (739785, 34)


In [1]:
import os

print(os.getcwd())
print(os.path.exists("earthquakes_5years_processed.csv"))

/Users/shilpathamaraiselvan
True


In [2]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)

NameError: name 'df' is not defined

In [3]:
import pandas as pd
import os

file_path = "/Users/shilpathamaraiselvan/earthquakes_5years_processed.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset loaded successfully!
Shape: (739785, 34)
Columns: ['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag', 'magType', 'place', 'status', 'tsunami', 'sig', 'net', 'nst', 'dmin', 'rms', 'gap', 'magError', 'depthError', 'magNst', 'locationSource', 'magSource', 'types', 'ids', 'sources', 'type', 'year', 'month', 'day', 'day_of_week', 'location_region', 'depth_category', 'strong_earthquake', 'major_earthquake']


In [4]:
%pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 24.0 kB/s  0:17:48 eta 0:00:040:00:25
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd

file_path = "/Users/shilpathamaraiselvan/earthquakes_5years_processed.csv"
df = pd.read_csv(file_path)

print("Shape:", df.shape)

Shape: (739785, 34)


In [2]:
import mysql.connector

print("MySQL connector imported successfully!")

MySQL connector imported successfully!


In [3]:
import mysql.connector
print("MySQL connector imported successfully!")

MySQL connector imported successfully!


In [4]:

import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=input("Enter your MySQL root password: "),
    database="global_seismic_trends"
)

print("Connected to MySQL successfully!")


Enter your MySQL root password:  Welcome123


Connected to MySQL successfully!


In [5]:
import pandas as pd

# Convert timestamps to UTC and remove timezone information
# because our MySQL columns use DATETIME(3).
for col in ["time", "updated"]:
    df[col] = pd.to_datetime(df[col], utc=True).dt.tz_localize(None)

# Check the data before importing
print("Dataset shape:", df.shape)
print("Missing values in time:", df["time"].isna().sum())
print("Missing values in updated:", df["updated"].isna().sum())
print(df[["id", "time", "updated", "mag", "depth_km"]].head())

ValueError: time data "2021-10-31 02:05:38+00:00" doesn't match format "%Y-%m-%d %H:%M:%S.%f%z", at position 346. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [6]:
import pandas as pd

# Convert timestamp strings to UTC, handling mixed formats
for col in ["time", "updated"]:
    df[col] = pd.to_datetime(
        df[col],
        format="mixed",
        utc=True,
        errors="coerce"
    ).dt.tz_localize(None)

print("Dataset shape:", df.shape)
print("Missing time values:", df["time"].isna().sum())
print("Missing updated values:", df["updated"].isna().sum())

print(df[["id", "time", "updated", "mag", "depth_km"]].head())

Dataset shape: (739785, 34)
Missing time values: 0
Missing updated values: 0
           id                    time                 updated   mag  depth_km
0  us7000fqcd 2021-10-31 23:47:49.663 2022-01-14 17:58:48.040  4.40    570.45
1  nc73647215 2021-10-31 23:42:49.670 2021-11-01 01:23:16.241  0.85      1.47
2  us7000fqca 2021-10-31 23:39:39.089 2022-01-14 17:58:48.040  4.40    496.53
3  hv72777307 2021-10-31 23:37:23.860 2021-11-03 04:20:31.790  1.52     33.67
4  ci39845863 2021-10-31 23:34:04.820 2021-11-01 23:48:09.591  1.46      9.14


In [7]:

import pandas as pd
import numpy as np
from datetime import datetime

# Confirm the destination table is empty before importing
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM earthquakes")
existing_rows = cursor.fetchone()[0]

if existing_rows != 0:
    print(f"Import stopped: the table already contains {existing_rows} rows.")
else:
    columns = df.columns.tolist()

    column_sql = ", ".join(f"`{col}`" for col in columns)
    placeholders = ", ".join(["%s"] * len(columns))

    insert_sql = f"""
        INSERT INTO earthquakes ({column_sql})
        VALUES ({placeholders})
    """

    def mysql_value(value):
        if pd.isna(value):
            return None
        if isinstance(value, pd.Timestamp):
            return value.to_pydatetime()
        if isinstance(value, np.generic):
            return value.item()
        return value

    batch_size = 2000
    total_rows = len(df)

    try:
        for start in range(0, total_rows, batch_size):
            batch = df.iloc[start:start + batch_size]
            records = [
                tuple(mysql_value(value) for value in row)
                for row in batch.itertuples(index=False, name=None)
            ]

            cursor.executemany(insert_sql, records)
            conn.commit()

            if (start + len(batch)) % 20000 == 0:
                print(f"Imported {start + len(batch):,} of {total_rows:,} rows")

        print(f"Import completed: {total_rows:,} rows processed.")

    except Exception:
        conn.rollback()
        print("An error occurred. Please share the error message.")
        raise

    finally:
        cursor.close()


Imported 20,000 of 739,785 rows
Imported 40,000 of 739,785 rows
Imported 60,000 of 739,785 rows
Imported 80,000 of 739,785 rows
Imported 100,000 of 739,785 rows
Imported 120,000 of 739,785 rows
Imported 140,000 of 739,785 rows
Imported 160,000 of 739,785 rows
Imported 180,000 of 739,785 rows
Imported 200,000 of 739,785 rows
Imported 220,000 of 739,785 rows
Imported 240,000 of 739,785 rows
Imported 260,000 of 739,785 rows
Imported 280,000 of 739,785 rows
Imported 300,000 of 739,785 rows
Imported 320,000 of 739,785 rows
Imported 340,000 of 739,785 rows
Imported 360,000 of 739,785 rows
Imported 380,000 of 739,785 rows
Imported 400,000 of 739,785 rows
Imported 420,000 of 739,785 rows
Imported 440,000 of 739,785 rows
Imported 460,000 of 739,785 rows
Imported 480,000 of 739,785 rows
Imported 500,000 of 739,785 rows
Imported 520,000 of 739,785 rows
Imported 540,000 of 739,785 rows
Imported 560,000 of 739,785 rows
Imported 580,000 of 739,785 rows
Imported 600,000 of 739,785 rows
Imported 620,0

In [8]:
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM earthquakes")
print("Rows in MySQL:", cursor.fetchone()[0])
cursor.close()

Rows in MySQL: 739785


True

In [9]:
cursor = conn.cursor()

cursor.execute("""
    SELECT id, time, place, mag, depth_km
    FROM earthquakes
    ORDER BY time DESC
    LIMIT 10
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

('ci41351583', datetime.datetime(2026, 10, 24, 6, 31, 36, 660000), '10 km ENE of Borrego Springs, CA', 1.09, 7.37)
('aka2026vbfltq', datetime.datetime(2026, 10, 24, 6, 29, 15, 297000), '71 km E of Port Alsworth, Alaska', 2.1, 151.5)
('ci41351575', datetime.datetime(2026, 10, 24, 6, 25, 31, 370000), '5 km ESE of Palomar Observatory, CA', 0.49, 6.4)
('nc75454146', datetime.datetime(2026, 10, 24, 6, 10, 24, 280000), '2 km WSW of Campbell, CA', 1.5, 3.63000011444092)
('us6000u4kj', datetime.datetime(2026, 10, 24, 6, 4, 16, 367000), '174 km NW of Port-Olry, Vanuatu', 5.3, 10.0)
('aka2026vbenrf', datetime.datetime(2026, 10, 24, 6, 1, 30, 15000), '52 km SSW of Skwentna, Alaska', 1.2, 19.3)
('aka2026vbelld', datetime.datetime(2026, 10, 24, 5, 58, 56, 682000), '108 km NNE of Arctic Village, Alaska', 3.3, 10.4)
('hv75063136', datetime.datetime(2026, 10, 24, 5, 53, 17, 40000), '25 km W of Volcano, Hawaii', 2.24, -0.920000016689301)
('aka2026vbeduv', datetime.datetime(2026, 10, 24, 5, 50, 7, 34400

True

In [10]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        year,
        COUNT(*) AS total_earthquakes,
        ROUND(AVG(mag), 2) AS average_magnitude,
        MAX(mag) AS highest_magnitude
    FROM earthquakes
    WHERE mag IS NOT NULL
    GROUP BY year
    ORDER BY year;
""")

results = cursor.fetchall()

print("Year | Total Earthquakes | Avg Magnitude | Highest Magnitude")
for row in results:
    print(row)

cursor.close()

Year | Total Earthquakes | Avg Magnitude | Highest Magnitude
(2021, 26182, 1.76, 7.5)
(2022, 150730, 1.68, 7.6)
(2023, 151614, 1.65, 7.8)
(2024, 155890, 1.52, 7.5)
(2025, 140357, 1.7, 8.8)
(2026, 114957, 1.75, 7.8)


True

In [11]:

cursor = conn.cursor()

cursor.execute("""
    SELECT
        CASE
            WHEN mag < 3.0 THEN 'Minor (<3.0)'
            WHEN mag < 4.0 THEN 'Light (3.0-3.9)'
            WHEN mag < 5.0 THEN 'Moderate (4.0-4.9)'
            WHEN mag < 6.0 THEN 'Strong (5.0-5.9)'
            WHEN mag < 7.0 THEN 'Major (6.0-6.9)'
            ELSE 'Great (7.0+)'
        END AS magnitude_category,
        COUNT(*) AS earthquake_count
    FROM earthquakes
    WHERE mag IS NOT NULL
    GROUP BY magnitude_category
    ORDER BY MIN(mag);
""")

for row in cursor.fetchall():
    print(row)

cursor.close()


('Minor (<3.0)', 636036)
('Light (3.0-3.9)', 23135)
('Moderate (4.0-4.9)', 71439)
('Strong (5.0-5.9)', 8455)
('Major (6.0-6.9)', 593)
('Great (7.0+)', 72)


True

In [12]:

cursor = conn.cursor()

cursor.execute("""
    SELECT
        depth_category,
        COUNT(*) AS earthquake_count,
        ROUND(AVG(depth_km), 2) AS average_depth_km,
        ROUND(AVG(mag), 2) AS average_magnitude
    FROM earthquakes
    WHERE depth_km IS NOT NULL
    GROUP BY depth_category
    ORDER BY earthquake_count DESC;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()


('Shallow', 643848, 9.76, 1.52)
('Deep', 95937, 123.56, 2.6)


True

In [13]:

cursor = conn.cursor()

cursor.execute("""
    SELECT
        location_region,
        COUNT(*) AS earthquake_count,
        ROUND(AVG(mag), 2) AS average_magnitude
    FROM earthquakes
    WHERE location_region IS NOT NULL
      AND location_region <> ''
    GROUP BY location_region
    ORDER BY earthquake_count DESC
    LIMIT 10;
""")

print("Region | Earthquake Count | Average Magnitude")
for row in cursor.fetchall():
    print(row)

cursor.close()


Region | Earthquake Count | Average Magnitude
('Alaska', 250125, 1.44)
('CA', 195783, 1.04)
('Hawaii', 37684, 1.91)
('Texas', 25847, 1.58)
('Nevada', 23029, 1.0)
('Oklahoma', 22596, 0.5)
('Washington', 16212, 0.79)
('Puerto Rico', 14166, 2.49)
('Utah', 12607, 1.16)
('Montana', 11023, 0.9)


True

In [14]:

cursor = conn.cursor()

cursor.execute("""
    SELECT
        year,
        COUNT(*) AS total_earthquakes,
        SUM(CASE WHEN mag >= 6.0 THEN 1 ELSE 0 END)
            AS magnitude_6_plus,
        SUM(CASE WHEN mag >= 7.0 THEN 1 ELSE 0 END)
            AS magnitude_7_plus
    FROM earthquakes
    WHERE mag IS NOT NULL
    GROUP BY year
    ORDER BY year;
""")

print("Year | Total | Magnitude 6+ | Magnitude 7+")
for row in cursor.fetchall():
    print(row)

cursor.close()


Year | Total | Magnitude 6+ | Magnitude 7+
(2021, 26182, Decimal('30'), Decimal('3'))
(2022, 150730, Decimal('127'), Decimal('11'))
(2023, 151614, Decimal('147'), Decimal('19'))
(2024, 155890, Decimal('99'), Decimal('10'))
(2025, 140357, Decimal('145'), Decimal('16'))
(2026, 114957, Decimal('117'), Decimal('13'))


True

In [15]:

cursor = conn.cursor()

cursor.execute("""
    SELECT
        month,
        COUNT(*) AS total_earthquakes,
        ROUND(AVG(mag), 2) AS average_magnitude,
        SUM(CASE WHEN mag >= 6.0 THEN 1 ELSE 0 END)
            AS magnitude_6_plus
    FROM earthquakes
    WHERE mag IS NOT NULL
    GROUP BY month
    ORDER BY month;
""")

print("Month | Total | Average Magnitude | Magnitude 6+")
for row in cursor.fetchall():
    print(row)

cursor.close()


Month | Total | Average Magnitude | Magnitude 6+
(1, 62389, 1.71, Decimal('58'))
(2, 60308, 1.59, Decimal('39'))
(3, 66547, 1.59, Decimal('63'))
(4, 63104, 1.6, Decimal('54'))
(5, 60629, 1.59, Decimal('59'))
(6, 59911, 1.61, Decimal('51'))
(7, 67168, 1.7, Decimal('64'))
(8, 63895, 1.67, Decimal('50'))
(9, 59761, 1.65, Decimal('53'))
(10, 57009, 1.69, Decimal('64'))
(11, 55353, 1.71, Decimal('58'))
(12, 63656, 1.79, Decimal('52'))


True

In [16]:

cursor = conn.cursor()

cursor.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT id) AS unique_earthquake_ids
    FROM earthquakes;
""")

print(cursor.fetchone())
cursor.close()


(739785, 739785)


True

In [17]:

cursor = conn.cursor()

cursor.execute("""
    SELECT id, time, place, mag, depth_km
    FROM earthquakes
    WHERE mag IS NOT NULL
    ORDER BY mag DESC, time DESC
    LIMIT 10;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()


('us6000qw60', datetime.datetime(2025, 7, 29, 23, 24, 52, 483000), '2025 Kamchatka Peninsula, Russia Earthquake', 8.8, 35.0)
('us6000tkt2', datetime.datetime(2026, 8, 14, 21, 58, 21, 561000), '66 km NNW of Ende, Indonesia', 7.8, 10.0)
('us7000srb1', datetime.datetime(2026, 6, 7, 23, 37, 41, 970000), '25 km SW of Kablalan, Philippines', 7.8, 56.991)
('us7000qx2g', datetime.datetime(2025, 9, 18, 18, 58, 14, 939000), '140 km E of Petropavlovsk-Kamchatsky, Russia', 7.8, 27.0)
('us6000jllz', datetime.datetime(2023, 2, 6, 1, 17, 34, 342000), 'Pazarcik earthquake, Kahramanmaras earthquake sequence', 7.8, 10.0)
('us6000u18k', datetime.datetime(2026, 10, 9, 17, 56, 6, 36000), '12 km WSW of Pitaloza Arriba, Panama', 7.7, 12.647)
('us7000pn9s', datetime.datetime(2025, 3, 28, 6, 20, 52, 715000), '2025 Mandalay, Burma (Myanmar) Earthquake', 7.7, 10.0)
('us6000kd0n', datetime.datetime(2023, 5, 19, 2, 57, 3, 172000), 'southeast of the Loyalty Islands', 7.7, 18.053)
('us6000rtdt', datetime.datetime(20

True

In [18]:
SELECT id, time, place, mag, depth_km
FROM earthquakes
WHERE mag IS NOT NULL
ORDER BY mag DESC, time DESC
LIMIT 10;

SyntaxError: invalid syntax (4224616126.py, line 1)

In [19]:

cursor = conn.cursor()

cursor.execute("""
    SELECT id, time, place, mag, depth_km
    FROM earthquakes
    WHERE depth_km IS NOT NULL
    ORDER BY depth_km DESC, time DESC
    LIMIT 10;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()


('us6000sp3p', datetime.datetime(2026, 4, 2, 16, 59, 17, 870000), '206 km ENE of Sola, Vanuatu', 4.0, 683.578)
('us6000k2db', datetime.datetime(2023, 4, 1, 18, 9, 17, 114000), '208 km ENE of Sola, Vanuatu', 4.0, 681.238)
('us7000kxdn', datetime.datetime(2023, 9, 18, 15, 35, 27, 270000), 'Vanuatu region', 4.2, 675.265)
('us6000mivr', datetime.datetime(2024, 3, 8, 22, 42, 23, 713000), 'Fiji region', 4.2, 671.043)
('us6000rk66', datetime.datetime(2025, 10, 29, 17, 7, 34, 157000), '205 km ESE of Levuka, Fiji', 4.8, 669.556)
('us7000q1jk', datetime.datetime(2025, 5, 10, 2, 3, 16, 756000), '299 km E of Levuka, Fiji', 4.2, 667.237)
('us7000s0st', datetime.datetime(2026, 2, 14, 8, 27, 48, 418000), '205 km ESE of Levuka, Fiji', 4.3, 667.197)
('us6000ta9m', datetime.datetime(2026, 7, 5, 14, 22, 25, 70000), '274 km SE of Levuka, Fiji', 5.7, 665.993)
('us6000tknp', datetime.datetime(2026, 8, 10, 16, 50, 9, 877000), '178 km NE of Sola, Vanuatu', 4.0, 665.326)
('us6000txkp', datetime.datetime(2026, 

True

In [20]:
SELECT id, time, place, mag, depth_km
FROM earthquakes
WHERE depth_km IS NOT NULL
ORDER BY depth_km DESC, time DESC
LIMIT 10;

SyntaxError: invalid syntax (1376230036.py, line 1)

In [21]:

cursor = conn.cursor()

cursor.execute("""
    SELECT id, time, place, mag, depth_km
    FROM earthquakes
    WHERE depth_km < 50
      AND mag > 7.5
    ORDER BY mag DESC;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()


('us6000qw60', datetime.datetime(2025, 7, 29, 23, 24, 52, 483000), '2025 Kamchatka Peninsula, Russia Earthquake', 8.8, 35.0)
('us6000jllz', datetime.datetime(2023, 2, 6, 1, 17, 34, 342000), 'Pazarcik earthquake, Kahramanmaras earthquake sequence', 7.8, 10.0)
('us6000tkt2', datetime.datetime(2026, 8, 14, 21, 58, 21, 561000), '66 km NNW of Ende, Indonesia', 7.8, 10.0)
('us7000qx2g', datetime.datetime(2025, 9, 18, 18, 58, 14, 939000), '140 km E of Petropavlovsk-Kamchatsky, Russia', 7.8, 27.0)
('us6000kd0n', datetime.datetime(2023, 5, 19, 2, 57, 3, 172000), 'southeast of the Loyalty Islands', 7.7, 18.053)
('us6000u18k', datetime.datetime(2026, 10, 9, 17, 56, 6, 36000), '12 km WSW of Pitaloza Arriba, Panama', 7.7, 12.647)
('us7000pn9s', datetime.datetime(2025, 3, 28, 6, 20, 52, 715000), '2025 Mandalay, Burma (Myanmar) Earthquake', 7.7, 10.0)
('us6000rgf4', datetime.datetime(2025, 10, 10, 20, 29, 20, 75000), 'Drake Passage', 7.6, 5.639)
('us6000rtdt', datetime.datetime(2025, 12, 8, 14, 15, 9

True

In [22]:
cursor = conn.cursor()

cursor.execute("""
    SELECT id, time, place, mag, depth_km
    FROM earthquakes
    WHERE depth_km < 50
      AND mag > 7.5
    ORDER BY mag DESC;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

('us6000qw60', datetime.datetime(2025, 7, 29, 23, 24, 52, 483000), '2025 Kamchatka Peninsula, Russia Earthquake', 8.8, 35.0)
('us6000jllz', datetime.datetime(2023, 2, 6, 1, 17, 34, 342000), 'Pazarcik earthquake, Kahramanmaras earthquake sequence', 7.8, 10.0)
('us6000tkt2', datetime.datetime(2026, 8, 14, 21, 58, 21, 561000), '66 km NNW of Ende, Indonesia', 7.8, 10.0)
('us7000qx2g', datetime.datetime(2025, 9, 18, 18, 58, 14, 939000), '140 km E of Petropavlovsk-Kamchatsky, Russia', 7.8, 27.0)
('us6000kd0n', datetime.datetime(2023, 5, 19, 2, 57, 3, 172000), 'southeast of the Loyalty Islands', 7.7, 18.053)
('us6000u18k', datetime.datetime(2026, 10, 9, 17, 56, 6, 36000), '12 km WSW of Pitaloza Arriba, Panama', 7.7, 12.647)
('us7000pn9s', datetime.datetime(2025, 3, 28, 6, 20, 52, 715000), '2025 Mandalay, Burma (Myanmar) Earthquake', 7.7, 10.0)
('us6000rgf4', datetime.datetime(2025, 10, 10, 20, 29, 20, 75000), 'Drake Passage', 7.6, 5.639)
('us6000rtdt', datetime.datetime(2025, 12, 8, 14, 15, 9

True

In [23]:
cursor = conn.cursor()

cursor.execute("""
    SELECT location_region, COUNT(*) AS earthquake_count
    FROM earthquakes
    GROUP BY location_region
    ORDER BY earthquake_count DESC
    LIMIT 20;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

('Alaska', 250125)
('CA', 195783)
('Hawaii', 37684)
('Texas', 25847)
('Nevada', 23029)
('Oklahoma', 22596)
('Washington', 16212)
('Puerto Rico', 14166)
('Utah', 12607)
('Montana', 11023)
('New Mexico', 11020)
('Indonesia', 8406)
('Russia', 6252)
('California', 5263)
('Philippines', 4820)
('Japan', 4714)
('Idaho', 4689)
('Wyoming', 4483)
('Papua New Guinea', 3395)
('Oregon', 3310)


True

In [24]:
cursor = conn.cursor()

cursor.execute("""
    SELECT place, latitude, longitude, depth_km
    FROM earthquakes
    WHERE place IS NOT NULL
    ORDER BY time DESC
    LIMIT 20;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

('10 km ENE of Borrego Springs, CA', 33.2905, -116.271333333333, 7.37)
('71 km E of Port Alsworth, Alaska', 60.176, -153.031, 151.5)
('5 km ESE of Palomar Observatory, CA', 33.3343333333333, -116.813, 6.4)
('2 km WSW of Campbell, CA', 37.2828330993652, -121.977333068848, 3.63000011444092)
('174 km NW of Port-Olry, Vanuatu', -14.1278, 165.7535, 10.0)
('52 km SSW of Skwentna, Alaska', 61.571, -151.835, 19.3)
('108 km NNE of Arctic Village, Alaska', 69.069, -144.851, 10.4)
('25 km W of Volcano, Hawaii', 19.4738330841064, -155.470993041992, -0.920000016689301)
('42 km NNW of Petersville, Alaska', 62.865, -150.969, 106.0)
('8 km WNW of Cobb, CA', 38.8393325805664, -122.808670043945, 2.29999995231628)
('175 km NW of Port-Olry, Vanuatu', -14.051, 165.8047, 10.0)
('184 km NW of Port-Olry, Vanuatu', -13.9786, 165.7483, 10.0)
('182 km NW of Port-Olry, Vanuatu', -14.0833, 165.6982, 10.0)
('29 km N of Fox River, Alaska', 60.119, -150.858, 45.9)
('93 km WNW of Yakutat, Alaska', 59.815, -141.305, 10

True

In [25]:
try:
    import geopandas as gpd
    print("GeoPandas is already installed.")
except ImportError:
    print("GeoPandas is not installed.")

GeoPandas is not installed.


In [27]:
%pip install geopandas

Note: you may need to restart the kernel to use updated packages.


In [1]:
import geopandas as gpd

print("GeoPandas version:", gpd.__version__)

GeoPandas version: 1.2.0


In [ ]:
import geopandas as gpd

world = gpd.read_file(
    "https://naturalearth.github.io/natural-earth-110m-countries/ne_110m_admin_0_countries.zip"
)

print(world.columns.tolist())
print(world.head())

In [3]:
import requests

url = "https://naturalearth.github.io/natural-earth-110m-countries/ne_110m_admin_0_countries.zip"

response = requests.get(url, timeout=20)

print("Status code:", response.status_code)
print("Downloaded bytes:", len(response.content))

Status code: 404
Downloaded bytes: 9115


In [4]:
import geopandas as gpd

url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"

world = gpd.read_file(url)

print("Countries loaded:", len(world))
print(world.columns.tolist())

Countries loaded: 177
['featurecla', 'scalerank', 'LABELRANK', 'SOVEREIGNT', 'SOV_A3', 'ADM0_DIF', 'LEVEL', 'TYPE', 'TLC', 'ADMIN', 'ADM0_A3', 'GEOU_DIF', 'GEOUNIT', 'GU_A3', 'SU_DIF', 'SUBUNIT', 'SU_A3', 'BRK_DIFF', 'NAME', 'NAME_LONG', 'BRK_A3', 'BRK_NAME', 'BRK_GROUP', 'ABBREV', 'POSTAL', 'FORMAL_EN', 'FORMAL_FR', 'NAME_CIAWF', 'NOTE_ADM0', 'NOTE_BRK', 'NAME_SORT', 'NAME_ALT', 'MAPCOLOR7', 'MAPCOLOR8', 'MAPCOLOR9', 'MAPCOLOR13', 'POP_EST', 'POP_RANK', 'POP_YEAR', 'GDP_MD', 'GDP_YEAR', 'ECONOMY', 'INCOME_GRP', 'FIPS_10', 'ISO_A2', 'ISO_A2_EH', 'ISO_A3', 'ISO_A3_EH', 'ISO_N3', 'ISO_N3_EH', 'UN_A3', 'WB_A2', 'WB_A3', 'WOE_ID', 'WOE_ID_EH', 'WOE_NOTE', 'ADM0_ISO', 'ADM0_DIFF', 'ADM0_TLC', 'ADM0_A3_US', 'ADM0_A3_FR', 'ADM0_A3_RU', 'ADM0_A3_ES', 'ADM0_A3_CN', 'ADM0_A3_TW', 'ADM0_A3_IN', 'ADM0_A3_NP', 'ADM0_A3_PK', 'ADM0_A3_DE', 'ADM0_A3_GB', 'ADM0_A3_BR', 'ADM0_A3_IL', 'ADM0_A3_PS', 'ADM0_A3_SA', 'ADM0_A3_EG', 'ADM0_A3_MA', 'ADM0_A3_PT', 'ADM0_A3_AR', 'ADM0_A3_JP', 'ADM0_A3_KO', 'ADM0_A3_

In [5]:
import pandas as pd
import geopandas as gpd

# Keep only records with valid coordinates and depth
earthquake_points = df[
    df["latitude"].between(-90, 90)
    & df["longitude"].between(-180, 180)
    & df["depth_km"].notna()
].copy()

print("Earthquakes available for mapping:", len(earthquake_points))
print(earthquake_points[["latitude", "longitude", "depth_km"]].head())

NameError: name 'df' is not defined

In [6]:
√

SyntaxError: invalid character '√' (U+221A) (1588554183.py, line 1)

In [7]:
import pandas as pd

df = pd.read_csv("earthquakes_5years_processed.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print(df.columns.tolist())

Dataset loaded successfully!
Shape: (739785, 34)
['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag', 'magType', 'place', 'status', 'tsunami', 'sig', 'net', 'nst', 'dmin', 'rms', 'gap', 'magError', 'depthError', 'magNst', 'locationSource', 'magSource', 'types', 'ids', 'sources', 'type', 'year', 'month', 'day', 'day_of_week', 'location_region', 'depth_category', 'strong_earthquake', 'major_earthquake']


In [8]:
import geopandas as gpd

earthquake_points = df[
    df["latitude"].between(-90, 90)
    & df["longitude"].between(-180, 180)
    & df["depth_km"].notna()
].copy()

print("Earthquakes available for mapping:", len(earthquake_points))
print(earthquake_points[["latitude", "longitude", "depth_km"]].head())

Earthquakes available for mapping: 739785
    latitude   longitude  depth_km
0 -17.942300 -178.463400    570.45
1  38.790833 -122.760834      1.47
2 -25.318900  179.162600    496.53
3  19.201667 -155.412833     33.67
4  35.711167 -117.583833      9.14


In [9]:
# Keep the country boundaries and continent labels
world_map = world[["CONTINENT", "geometry"]].copy()

# Ensure both datasets use the same coordinate system
world_map = world_map.set_crs("EPSG:4326", allow_override=True)

continent_results = []

batch_size = 50000

for start in range(0, len(earthquake_points), batch_size):
    batch = earthquake_points.iloc[start:start + batch_size].copy()

    points = gpd.GeoDataFrame(
        batch,
        geometry=gpd.points_from_xy(batch["longitude"], batch["latitude"]),
        crs="EPSG:4326"
    )

    matched = gpd.sjoin(
        points,
        world_map,
        how="left",
        predicate="within"
    )

    continent_results.append(
        matched[["depth_km", "CONTINENT"]]
    )

    print("Processed", min(start + batch_size, len(earthquake_points)),
          "of", len(earthquake_points))

continent_data = pd.concat(continent_results, ignore_index=True)

print("\nContinent mapping completed!")
print(continent_data["CONTINENT"].value_counts(dropna=False))

Processed 50000 of 739785
Processed 100000 of 739785
Processed 150000 of 739785
Processed 200000 of 739785
Processed 250000 of 739785
Processed 300000 of 739785
Processed 350000 of 739785
Processed 400000 of 739785
Processed 450000 of 739785
Processed 500000 of 739785
Processed 550000 of 739785
Processed 600000 of 739785
Processed 650000 of 739785
Processed 700000 of 739785
Processed 739785 of 739785

Continent mapping completed!
CONTINENT
North America    541748
NaN              180012
Asia               9282
South America      5028
Oceania            1620
Europe             1331
Africa              761
Antarctica            3
Name: count, dtype: int64


In [10]:
average_depth_by_continent = (
    continent_data
    .dropna(subset=["CONTINENT"])
    .groupby("CONTINENT")
    .agg(
        earthquake_count=("depth_km", "count"),
        average_depth_km=("depth_km", "mean")
    )
    .reset_index()
    .sort_values("average_depth_km", ascending=False)
)

average_depth_by_continent["average_depth_km"] = (
    average_depth_by_continent["average_depth_km"].round(2)
)

print(average_depth_by_continent.to_string(index=False))

    CONTINENT  earthquake_count  average_depth_km
South America              5028            124.11
      Oceania              1620             71.83
         Asia              9282             45.15
       Europe              1331             33.47
North America            541748             14.58
   Antarctica                 3             10.00
       Africa               761              9.77


In [11]:
average_depth_by_continent.to_csv(
    "average_depth_by_continent.csv",
    index=False
)

print("Saved: average_depth_by_continent.csv")

Saved: average_depth_by_continent.csv


In [12]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        magType,
        COUNT(*) AS earthquake_count,
        ROUND(AVG(mag), 2) AS average_magnitude
    FROM earthquakes
    WHERE magType IS NOT NULL
      AND mag IS NOT NULL
    GROUP BY magType
    ORDER BY average_magnitude DESC;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

NameError: name 'conn' is not defined

In [13]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=input("Enter your MySQL password: "),
    database="global_seismic_trends"
)

print("MySQL connected successfully:", conn.is_connected())

Enter your MySQL password:  Welcome123


MySQL connected successfully: True


In [14]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        magType,
        COUNT(*) AS earthquake_count,
        ROUND(AVG(mag), 2) AS average_magnitude
    FROM earthquakes
    WHERE magType IS NOT NULL
      AND mag IS NOT NULL
    GROUP BY magType
    ORDER BY average_magnitude DESC;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

('mwc', 1, 7.0)
('Mi', 1, 6.6)
('ms_20', 1, 5.8)
('mwb', 26, 5.76)
('mww', 6516, 5.35)
('mwp', 1, 5.25)
('ms_vx', 5, 4.5)
('mb', 73357, 4.42)
('mwr', 2516, 4.31)
('mw', 548, 3.92)
('mlr', 8, 3.49)
('mlg', 1, 3.3)
('mb_lg', 642, 2.6)
('ml(texnet)', 1068, 1.99)
('ml', 486003, 1.32)
('md', 168481, 1.25)
('mlv', 54, 0.85)
('mh', 475, 0.31)
('mun', 26, 0.0)


True

In [15]:
magtype_results = pd.DataFrame(
    [
        ("mwc", 1, 7.0),
        ("Mi", 1, 6.6),
        ("ms_20", 1, 5.8),
        ("mwb", 26, 5.76),
        ("mww", 6516, 5.35),
        ("mwp", 1, 5.25),
        ("ms_vx", 5, 4.5),
        ("mb", 73357, 4.42),
        ("mwr", 2516, 4.31),
        ("mw", 548, 3.92),
        ("mlr", 8, 3.49),
        ("mlg", 1, 3.3),
        ("mb_lg", 642, 2.6),
        ("ml(texnet)", 1068, 1.99),
        ("ml", 486003, 1.32),
        ("md", 168481, 1.25),
        ("mlv", 54, 0.85),
        ("mh", 475, 0.31),
        ("mun", 26, 0.0),
    ],
    columns=["magType", "earthquake_count", "average_magnitude"]
)

magtype_results.to_csv("average_magnitude_by_magtype.csv", index=False)

print("Saved: average_magnitude_by_magtype.csv")

Saved: average_magnitude_by_magtype.csv


In [16]:
cursor = conn.cursor()

cursor.execute("""
    SELECT year, COUNT(*) AS earthquake_count
    FROM earthquakes
    GROUP BY year
    ORDER BY earthquake_count DESC;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

(2024, 155899)
(2023, 151618)
(2022, 150740)
(2025, 140366)
(2026, 114972)
(2021, 26190)


True

In [17]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        month,
        COUNT(*) AS earthquake_count
    FROM earthquakes
    GROUP BY month
    ORDER BY earthquake_count DESC;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

(7, 67170)
(3, 66551)
(8, 63902)
(12, 63666)
(4, 63106)
(1, 62398)
(5, 60633)
(2, 60312)
(6, 59912)
(9, 59762)
(10, 57011)
(11, 55362)


True

In [18]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        day_of_week,
        COUNT(*) AS earthquake_count
    FROM earthquakes
    GROUP BY day_of_week
    ORDER BY earthquake_count DESC;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

('Thursday', 106951)
('Wednesday', 106872)
('Monday', 106046)
('Friday', 106020)
('Tuesday', 105654)
('Sunday', 104482)
('Saturday', 103760)


True

In [19]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        HOUR(time) AS hour_of_day,
        COUNT(*) AS earthquake_count
    FROM earthquakes
    GROUP BY HOUR(time)
    ORDER BY hour_of_day;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

(0, 30578)
(1, 30211)
(2, 30873)
(3, 31006)
(4, 31375)
(5, 31482)
(6, 31685)
(7, 32051)
(8, 32407)
(9, 32226)
(10, 31983)
(11, 31024)
(12, 30754)
(13, 30691)
(14, 29677)
(15, 29919)
(16, 30178)
(17, 30627)
(18, 30488)
(19, 30953)
(20, 30283)
(21, 29944)
(22, 29670)
(23, 29700)


True

In [20]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        net,
        COUNT(*) AS earthquake_count
    FROM earthquakes
    WHERE net IS NOT NULL
    GROUP BY net
    ORDER BY earthquake_count DESC
    LIMIT 10;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

('ak', 177240)
('nc', 113981)
('us', 109461)
('ci', 84105)
('av', 57664)
('hv', 37600)
('tx', 36925)
('nn', 28564)
('ok', 22626)
('uw', 20762)


True

In [21]:
cursor = conn.cursor()

cursor.execute("SHOW COLUMNS FROM earthquakes")

for row in cursor.fetchall():
    print(row[0])

cursor.close()

id
time
updated
latitude
longitude
depth_km
mag
magType
place
status
tsunami
sig
net
nst
dmin
rms
gap
magError
depthError
magNst
locationSource
magSource
types
ids
sources
type
year
month
day
day_of_week
location_region
depth_category
strong_earthquake
major_earthquake


True

In [22]:
cursor = conn.cursor()

cursor.execute("SHOW COLUMNS FROM earthquakes")

columns = [row[0] for row in cursor.fetchall()]

print("Casualty field available:", "casualties" in columns)
print("Economic loss field available:", "economic_loss" in columns)

cursor.close()

Casualty field available: False
Economic loss field available: False


True

In [23]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        status,
        COUNT(*) AS earthquake_count
    FROM earthquakes
    GROUP BY status
    ORDER BY earthquake_count DESC;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

('reviewed', 658081)
('automatic', 81704)


True

In [24]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        type,
        COUNT(*) AS event_count
    FROM earthquakes
    GROUP BY type
    ORDER BY event_count DESC;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

('earthquake', 722517)
('quarry blast', 7110)
('explosion', 5069)
('ice quake', 3701)
('mining explosion', 1043)
('other event', 263)
('Landslide', 42)
('volcanic eruption', 12)
('sonic boom', 9)
('chemical explosion', 6)
('experimental explosion', 4)
('mine collapse', 3)
('snow avalanche', 2)
('meteorite', 2)
('rockslide', 1)
('accidental explosion', 1)


True

In [25]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        types,
        COUNT(*) AS event_count
    FROM earthquakes
    WHERE types IS NOT NULL
    GROUP BY types
    ORDER BY event_count DESC
    LIMIT 20;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

(',origin,phase-data,', 501561)
(',nearby-cities,origin,phase-data,scitech-link,', 122118)
(',nearby-cities,origin,phase-data,', 37566)
(',focal-mechanism,nearby-cities,origin,phase-data,scitech-link,', 23939)
(',dyfi,origin,phase-data,', 17568)
(',earthquake-name,origin,phase-data,', 7522)
(',dyfi,focal-mechanism,nearby-cities,origin,phase-data,scitech-link,', 5753)
(',dyfi,nearby-cities,origin,phase-data,scitech-link,', 4934)
(',origin,phase-data,shakemap,', 2627)
(',focal-mechanism,nearby-cities,origin,phase-data,', 2177)
(',moment-tensor,origin,phase-data,', 1338)
(',dyfi,moment-tensor,origin,phase-data,', 1218)
(',dyfi,origin,phase-data,shakemap,', 1071)
(',dyfi,nearby-cities,origin,phase-data,', 631)
(',internal-moment-tensor,origin,phase-data,', 518)
(',internal-moment-tensor,losspager,moment-tensor,origin,phase-data,shakemap,', 485)
(',dyfi,moment-tensor,origin,phase-data,shakemap,', 445)
(',internal-moment-tensor,moment-tensor,origin,phase-data,', 418)
(',dyfi,internal-moment-

True

In [26]:
# Add RMS and gap values to the existing continent mapping
quality_data = continent_data[["CONTINENT"]].copy()

quality_data["rms"] = earthquake_points["rms"].reset_index(drop=True)
quality_data["gap"] = earthquake_points["gap"].reset_index(drop=True)

quality_summary = (
    quality_data
    .dropna(subset=["CONTINENT"])
    .groupby("CONTINENT")
    .agg(
        earthquake_count=("CONTINENT", "size"),
        average_rms=("rms", "mean"),
        average_gap=("gap", "mean")
    )
    .reset_index()
)

quality_summary[["average_rms", "average_gap"]] = (
    quality_summary[["average_rms", "average_gap"]].round(2)
)

print(quality_summary.to_string(index=False))

    CONTINENT  earthquake_count  average_rms  average_gap
       Africa               761         0.68       116.98
   Antarctica                 3         0.47        87.00
         Asia              9282         0.71        93.71
       Europe              1331         0.72        88.63
North America            541748         0.24       102.97
      Oceania              1620         0.70        94.79
South America              5028         0.75        88.75


In [27]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        id,
        time,
        place,
        mag,
        nst
    FROM earthquakes
    WHERE nst > 20
    ORDER BY nst DESC
    LIMIT 20;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

('us6000m12f', datetime.datetime(2024, 1, 2, 1, 17, 31, 568000), '11 km W of Anamizu, Japan', 5.4, 619)
('us6000qzfl', datetime.datetime(2025, 8, 9, 8, 1, 41, 17000), '120 km ENE of Ozernovskiy, Russia', 5.0, 566)
('us7000rluk', datetime.datetime(2026, 1, 1, 6, 46, 54, 742000), '111 km N of Yakutat, Alaska', 5.7, 516)
('us7000pvtr', datetime.datetime(2025, 4, 29, 14, 53, 37, 897000), 'Macquarie Island region', 6.8, 475)
('usd001097k', datetime.datetime(2023, 12, 11, 18, 35, 59, 877000), '49 km WNW of San Antonio de los Cobres, Argentina', 5.5, 466)
('us7000ilwt', datetime.datetime(2022, 11, 2, 4, 53, 13, 239000), 'North Pacific Ocean', 6.0, 452)
('us7000tgky', datetime.datetime(2026, 9, 11, 10, 47, 12, 115000), '149 km WSW of Adak, Alaska', 5.1, 445)
('us6000i6p3', datetime.datetime(2022, 7, 29, 22, 40, 59, 955000), 'Kermadec Islands region', 5.1, 444)
('us7000rptl', datetime.datetime(2026, 1, 15, 8, 52, 43, 219000), '280 km ESE of Attu Station, Alaska', 5.9, 437)
('us6000pxvx', dateti

True

In [28]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        year,
        COUNT(*) AS tsunami_event_count
    FROM earthquakes
    WHERE tsunami = 1
    GROUP BY year
    ORDER BY year;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()


(2021, 25)
(2022, 136)
(2023, 119)
(2024, 114)
(2025, 142)
(2026, 50)


True

In [29]:
cursor = conn.cursor()

cursor.execute("SHOW COLUMNS FROM earthquakes")

columns = [row[0] for row in cursor.fetchall()]

print("Alert field available:", "alert" in columns)

cursor.close()

Alert field available: False


True

In [30]:
# Check the available country and continent fields
print(world[["NAME", "CONTINENT"]].head(10))

                       NAME      CONTINENT
0                      Fiji        Oceania
1                  Tanzania         Africa
2                 W. Sahara         Africa
3                    Canada  North America
4  United States of America  North America
5                Kazakhstan           Asia
6                Uzbekistan           Asia
7          Papua New Guinea        Oceania
8                 Indonesia           Asia
9                 Argentina  South America


In [31]:
world_countries = world[["NAME", "CONTINENT", "geometry"]].copy()
world_countries = world_countries.set_crs(
    "EPSG:4326", allow_override=True
)

country_results = []

batch_size = 50000

for start in range(0, len(earthquake_points), batch_size):
    batch = earthquake_points.iloc[start:start + batch_size].copy()

    points = gpd.GeoDataFrame(
        batch,
        geometry=gpd.points_from_xy(
            batch["longitude"], batch["latitude"]
        ),
        crs="EPSG:4326"
    )

    matched = gpd.sjoin(
        points,
        world_countries,
        how="left",
        predicate="within"
    )

    country_results.append(
        matched[["mag", "NAME", "CONTINENT"]]
    )

    print(
        "Processed",
        min(start + batch_size, len(earthquake_points)),
        "of",
        len(earthquake_points)
    )

country_data = pd.concat(country_results, ignore_index=True)

print("\nCountry mapping completed!")
print("Mapped records:", country_data["NAME"].notna().sum())

Processed 50000 of 739785
Processed 100000 of 739785
Processed 150000 of 739785
Processed 200000 of 739785
Processed 250000 of 739785
Processed 300000 of 739785
Processed 350000 of 739785
Processed 400000 of 739785
Processed 450000 of 739785
Processed 500000 of 739785
Processed 550000 of 739785
Processed 600000 of 739785
Processed 650000 of 739785
Processed 700000 of 739785
Processed 739785 of 739785

Country mapping completed!
Mapped records: 559773


In [32]:
top_countries_by_magnitude = (
    country_data
    .dropna(subset=["NAME", "mag"])
    .groupby("NAME")
    .agg(
        earthquake_count=("mag", "count"),
        average_magnitude=("mag", "mean")
    )
    .query("earthquake_count >= 100")
    .reset_index()
    .sort_values("average_magnitude", ascending=False)
    .head(5)
)

top_countries_by_magnitude["average_magnitude"] = (
    top_countries_by_magnitude["average_magnitude"].round(2)
)

print(top_countries_by_magnitude.to_string(index=False))

            NAME  earthquake_count  average_magnitude
         Vanuatu               106               4.71
          Panama               105               4.62
     Philippines               589               4.58
Papua New Guinea              1112               4.55
         Iceland               134               4.53


In [33]:
top_countries_by_magnitude.to_csv(
    "top_5_countries_average_magnitude.csv",
    index=False
)

print("Saved: top_5_countries_average_magnitude.csv")

Saved: top_5_countries_average_magnitude.csv


In [34]:
# Attach depth and date fields to the country mapping
country_month_data = country_data[["NAME"]].copy()

country_month_data["depth_km"] = (
    earthquake_points["depth_km"].reset_index(drop=True)
)
country_month_data["year"] = (
    earthquake_points["year"].reset_index(drop=True)
)
country_month_data["month"] = (
    earthquake_points["month"].reset_index(drop=True)
)

country_month_data = country_month_data.dropna(subset=["NAME", "depth_km"])

country_month_data["depth_category"] = country_month_data["depth_km"].apply(
    lambda depth: "Shallow" if depth < 50 else "Deep"
)

country_month_summary = (
    country_month_data
    .groupby(["NAME", "year", "month"])["depth_category"]
    .nunique()
    .reset_index(name="depth_category_count")
)

both_depth_types = country_month_summary[
    country_month_summary["depth_category_count"] == 2
].sort_values(["NAME", "year", "month"])

print("Country-month combinations with both shallow and deep earthquakes:")
print(both_depth_types.to_string(index=False))
print("\nTotal combinations:", len(both_depth_types))

Country-month combinations with both shallow and deep earthquakes:
                    NAME  year  month  depth_category_count
             Afghanistan  2021     10                     2
             Afghanistan  2021     11                     2
             Afghanistan  2021     12                     2
             Afghanistan  2022      1                     2
             Afghanistan  2022      2                     2
             Afghanistan  2022      3                     2
             Afghanistan  2022      4                     2
             Afghanistan  2022      5                     2
             Afghanistan  2022      6                     2
             Afghanistan  2022      7                     2
             Afghanistan  2022      8                     2
             Afghanistan  2022      9                     2
             Afghanistan  2022     10                     2
             Afghanistan  2022     11                     2
             Afghanistan  2022   

In [35]:
country_month_counts = (
    both_depth_types
    .groupby("NAME")
    .size()
    .reset_index(name="months_with_both")
    .sort_values("months_with_both", ascending=False)
)

print("Total matching country-month combinations:", len(both_depth_types))
print("\nTop 10 countries:")
print(country_month_counts.head(10).to_string(index=False))

Total matching country-month combinations: 1114

Top 10 countries:
                    NAME  months_with_both
               Indonesia                61
United States of America                61
        Papua New Guinea                60
                  Mexico                59
                  Russia                58
                   Chile                57
             Philippines                57
             Puerto Rico                56
                   Japan                55
                    Peru                55


In [36]:
both_depth_types.to_csv(
    "countries_with_shallow_and_deep_same_month.csv",
    index=False
)

country_month_counts.to_csv(
    "country_month_depth_summary.csv",
    index=False
)

print("Question 22 results saved successfully.")

Question 22 results saved successfully.


In [37]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        year,
        COUNT(*) AS earthquake_count
    FROM earthquakes
    GROUP BY year
    ORDER BY year;
""")

yearly_counts = pd.DataFrame(
    cursor.fetchall(),
    columns=["year", "earthquake_count"]
)

cursor.close()

yearly_counts["yoy_growth_percent"] = (
    yearly_counts["earthquake_count"]
    .pct_change()
    .mul(100)
    .round(2)
)

print(yearly_counts.to_string(index=False))

 year  earthquake_count  yoy_growth_percent
 2021             26190                 NaN
 2022            150740              475.56
 2023            151618                0.58
 2024            155899                2.82
 2025            140366               -9.96
 2026            114972              -18.09


In [38]:
region_summary = (
    country_data
    .dropna(subset=["NAME", "mag"])
    .groupby("NAME")
    .agg(
        earthquake_count=("mag", "count"),
        average_magnitude=("mag", "mean")
    )
    .reset_index()
)

# Keep regions with at least 100 records for more stable averages
region_summary = region_summary[
    region_summary["earthquake_count"] >= 100
].copy()

# Normalize frequency and average magnitude to 0–1
region_summary["frequency_score"] = (
    region_summary["earthquake_count"] /
    region_summary["earthquake_count"].max()
)

region_summary["magnitude_score"] = (
    (region_summary["average_magnitude"] -
     region_summary["average_magnitude"].min()) /
    (region_summary["average_magnitude"].max() -
     region_summary["average_magnitude"].min())
)

# Give frequency and magnitude equal weight
region_summary["activity_score"] = (
    0.5 * region_summary["frequency_score"] +
    0.5 * region_summary["magnitude_score"]
)

top_3_regions = region_summary.sort_values(
    "activity_score", ascending=False
).head(3)

print(
    top_3_regions[
        ["NAME", "earthquake_count", "average_magnitude", "activity_score"]
    ].round(3).to_string(index=False)
)

                    NAME  earthquake_count  average_magnitude  activity_score
                 Vanuatu               106              4.708           0.500
United States of America            520719              1.145           0.500
                  Panama               105              4.615           0.487


In [39]:
top_3_regions.to_csv("top_3_seismically_active_regions.csv", index=False)
print("Saved: top_3_seismically_active_regions.csv")

Saved: top_3_seismically_active_regions.csv


In [40]:
equatorial_data = country_data[["NAME"]].copy()

equatorial_data["latitude"] = (
    earthquake_points["latitude"].reset_index(drop=True)
)
equatorial_data["depth_km"] = (
    earthquake_points["depth_km"].reset_index(drop=True)
)

equatorial_data = equatorial_data[
    equatorial_data["latitude"].between(-5, 5)
].dropna(subset=["NAME", "depth_km"])

equatorial_depth_summary = (
    equatorial_data
    .groupby("NAME")
    .agg(
        earthquake_count=("depth_km", "count"),
        average_depth_km=("depth_km", "mean")
    )
    .query("earthquake_count >= 10")
    .reset_index()
    .sort_values("average_depth_km", ascending=False)
)

equatorial_depth_summary["average_depth_km"] = (
    equatorial_depth_summary["average_depth_km"].round(2)
)

print(equatorial_depth_summary.head(20).to_string(index=False))

            NAME  earthquake_count  average_depth_km
Papua New Guinea               379             84.20
         Ecuador               224             80.20
            Peru                72             66.26
       Indonesia              1184             55.66
        Colombia               141             55.60
 Dem. Rep. Congo                29             10.16
        Tanzania                37             10.00
          Uganda                12             10.00


In [41]:
equatorial_depth_summary.to_csv(
    "average_depth_equatorial_countries.csv",
    index=False
)

print("Saved: average_depth_equatorial_countries.csv")

Saved: average_depth_equatorial_countries.csv


In [42]:
country_depth_counts = country_data[["NAME"]].copy()

country_depth_counts["depth_km"] = (
    earthquake_points["depth_km"].reset_index(drop=True)
)

country_depth_counts = country_depth_counts.dropna(
    subset=["NAME", "depth_km"]
)

country_depth_counts["shallow"] = (
    country_depth_counts["depth_km"] < 50
).astype(int)

country_depth_counts["deep"] = (
    country_depth_counts["depth_km"] >= 50
).astype(int)

shallow_deep_ratio = (
    country_depth_counts
    .groupby("NAME")
    .agg(
        shallow_count=("shallow", "sum"),
        deep_count=("deep", "sum")
    )
    .query("deep_count >= 10")
    .reset_index()
)

shallow_deep_ratio["shallow_to_deep_ratio"] = (
    shallow_deep_ratio["shallow_count"] /
    shallow_deep_ratio["deep_count"]
).round(2)

shallow_deep_ratio = shallow_deep_ratio.sort_values(
    "shallow_to_deep_ratio", ascending=False
)

print(shallow_deep_ratio.head(10).to_string(index=False))

                    NAME  shallow_count  deep_count  shallow_to_deep_ratio
                   China           1675          37                  45.27
             Puerto Rico           5972         182                  32.81
                   India            182          13                  14.00
United States of America         479687       41075                  11.68
                  Mexico           2173         199                  10.92
                  Greece            229          21                  10.90
                 Vanuatu             95          11                   8.64
                Pakistan            171          23                   7.43
               Venezuela             78          16                   4.88
              Costa Rica             53          20                   2.65


In [43]:
shallow_deep_ratio.to_csv(
    "countries_shallow_to_deep_ratio.csv",
    index=False
)

print("Saved: countries_shallow_to_deep_ratio.csv")

Saved: countries_shallow_to_deep_ratio.csv


In [44]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        tsunami,
        COUNT(*) AS earthquake_count,
        ROUND(AVG(mag), 2) AS average_magnitude
    FROM earthquakes
    WHERE tsunami IN (0, 1)
      AND mag IS NOT NULL
    GROUP BY tsunami
    ORDER BY tsunami;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

(0, 739144, 1.66)
(1, 586, 5.39)


True

In [45]:
tsunami_magnitude_results = pd.DataFrame(
    [
        (0, 739144, 1.66),
        (1, 586, 5.39)
    ],
    columns=["tsunami", "earthquake_count", "average_magnitude"]
)

tsunami_magnitude_results.to_csv(
    "tsunami_vs_non_tsunami_magnitude.csv",
    index=False
)

print("Saved: tsunami_vs_non_tsunami_magnitude.csv")

Saved: tsunami_vs_non_tsunami_magnitude.csv


In [46]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        id,
        time,
        place,
        mag,
        gap,
        rms
    FROM earthquakes
    WHERE gap > 180
       OR rms > 1.0
    ORDER BY gap DESC, rms DESC
    LIMIT 20;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

('uw61988777', datetime.datetime(2024, 4, 15, 19, 24, 33, 260000), '9 km N of Peaceful Valley, Washington', 1.06, 360.0, 0.5)
('ci40440272', datetime.datetime(2023, 4, 12, 23, 39, 52, 500000), '8km SE of Avalon, CA', 1.2, 360.0, 0.22)
('ci41277408', datetime.datetime(2025, 8, 28, 14, 59, 24, 990000), '10 km NNW of Thousand Palms, CA', 0.05, 360.0, 0.2)
('ci39623194', datetime.datetime(2023, 7, 26, 22, 30, 52, 470000), '8km SE of Avalon, CA', 1.17, 360.0, 0.19)
('nn00907449', datetime.datetime(2025, 9, 16, 3, 28, 29, 706000), '59 km NE of Valmy, Nevada', 2.13, 360.0, 0.1436)
('uw61950867', datetime.datetime(2023, 9, 14, 13, 42, 25, 760000), 'Washington', 0.53, 360.0, 0.08)
('uw61873031', datetime.datetime(2022, 8, 17, 7, 25, 7, 910000), '18 km ESE of Three Rivers, Oregon', -0.47, 360.0, 0.03)
('ci37542821', datetime.datetime(2025, 4, 10, 5, 24, 51, 660000), '12 km SSE of Bombay Beach, CA', 2.35, 359.0, 0.2)
('uw61803081', datetime.datetime(2021, 12, 5, 22, 7, 48, 100000), '2 km WNW of B

True

In [47]:
cursor = conn.cursor()

cursor.execute("""
    SELECT COUNT(*)
    FROM earthquakes
    WHERE gap > 180 OR rms > 1.0;
""")

total_flagged = cursor.fetchone()[0]
cursor.close()

print("Total potentially lower-reliability events:", total_flagged)

Total potentially lower-reliability events: 114652


In [48]:
print(df[["id", "time", "latitude", "longitude", "mag", "place"]].head())
print("\nMissing values:")
print(df[["id", "time", "latitude", "longitude"]].isnull().sum())

           id                              time   latitude   longitude   mag  \
0  us7000fqcd  2021-10-31 23:47:49.663000+00:00 -17.942300 -178.463400  4.40   
1  nc73647215  2021-10-31 23:42:49.670000+00:00  38.790833 -122.760834  0.85   
2  us7000fqca  2021-10-31 23:39:39.089000+00:00 -25.318900  179.162600  4.40   
3  hv72777307  2021-10-31 23:37:23.860000+00:00  19.201667 -155.412833  1.52   
4  ci39845863  2021-10-31 23:34:04.820000+00:00  35.711167 -117.583833  1.46   

                        place  
0    235 km E of Levuka, Fiji  
1  2km NNW of The Geysers, CA  
2   south of the Fiji Islands  
3    6 km E of P?hala, Hawaii  
4   13km NE of Ridgecrest, CA  

Missing values:
id           0
time         0
latitude     0
longitude    0
dtype: int64


In [50]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

earthquake_pairs_df = df[
    ["id", "time", "latitude", "longitude", "mag", "place"]
].copy()

# Handle timestamps with mixed formats and UTC offsets
earthquake_pairs_df["time"] = pd.to_datetime(
    earthquake_pairs_df["time"],
    format="mixed",
    utc=True,
    errors="coerce"
)

# Remove missing or invalid values
earthquake_pairs_df = earthquake_pairs_df.dropna(
    subset=["id", "time", "latitude", "longitude"]
)

# Keep valid geographic coordinates
earthquake_pairs_df = earthquake_pairs_df[
    earthquake_pairs_df["latitude"].between(-90, 90)
    & earthquake_pairs_df["longitude"].between(-180, 180)
].reset_index(drop=True)

print("Records ready for analysis:", len(earthquake_pairs_df))
print("Unique earthquake IDs:", earthquake_pairs_df["id"].nunique())
print("Time range:", earthquake_pairs_df["time"].min(),
      "to", earthquake_pairs_df["time"].max())

Records ready for analysis: 739785
Unique earthquake IDs: 739785
Time range: 2021-10-24 00:02:44.820000+00:00 to 2026-10-24 06:31:36.660000+00:00


In [1]:
# Convert latitude and longitude to radians
coordinates_rad = np.radians(
    earthquake_pairs_df[["latitude", "longitude"]].to_numpy()
)

# Build a spatial index
earthquake_tree = BallTree(coordinates_rad, metric="haversine")

# Earth's mean radius in kilometres
earth_radius_km = 6371.0088

# Search radius: 50 km
radius_rad = 50 / earth_radius_km

# Find nearby candidate events
nearby_indices = earthquake_tree.query_radius(
    coordinates_rad,
    r=radius_rad,
    return_distance=False
)

# Count candidate neighbours, excluding each event itself
candidate_count = sum(len(indices) - 1 for indices in nearby_indices)

print("Nearby candidate relationships:", candidate_count)
print("Spatial search completed successfully.")

NameError: name 'np' is not defined

In [2]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

print("Required libraries imported successfully.")

Required libraries imported successfully.


In [3]:
# Convert latitude and longitude to radians
coordinates_rad = np.radians(
    earthquake_pairs_df[["latitude", "longitude"]].to_numpy()
)

# Build a spatial index
earthquake_tree = BallTree(coordinates_rad, metric="haversine")

# Earth's mean radius in kilometres
earth_radius_km = 6371.0088

# Search radius: 50 km
radius_rad = 50 / earth_radius_km

# Find nearby candidate events
nearby_indices = earthquake_tree.query_radius(
    coordinates_rad,
    r=radius_rad,
    return_distance=False
)

# Count candidate neighbours, excluding each event itself
candidate_count = sum(len(indices) - 1 for indices in nearby_indices)

print("Nearby candidate relationships:", candidate_count)
print("Spatial search completed successfully.")

NameError: name 'earthquake_pairs_df' is not defined

In [4]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

# Recreate the working dataset from df
earthquake_pairs_df = df[
    ["id", "time", "latitude", "longitude", "mag", "place"]
].copy()

# Standardize timestamps
earthquake_pairs_df["time"] = pd.to_datetime(
    earthquake_pairs_df["time"],
    format="mixed",
    utc=True,
    errors="coerce"
)

# Remove invalid records
earthquake_pairs_df = earthquake_pairs_df.dropna(
    subset=["id", "time", "latitude", "longitude"]
)

earthquake_pairs_df = earthquake_pairs_df[
    earthquake_pairs_df["latitude"].between(-90, 90)
    & earthquake_pairs_df["longitude"].between(-180, 180)
].reset_index(drop=True)

print("Prepared records:", len(earthquake_pairs_df))

NameError: name 'df' is not defined

In [5]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

# Reload the processed dataset saved on your Mac
df = pd.read_csv(
    "/Users/shilpathamaraiselvan/earthquakes_5years_processed.csv"
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print(df[["id", "time", "latitude", "longitude", "mag", "place"]].head())

Dataset loaded successfully!
Shape: (739785, 34)
           id                              time   latitude   longitude   mag  \
0  us7000fqcd  2021-10-31 23:47:49.663000+00:00 -17.942300 -178.463400  4.40   
1  nc73647215  2021-10-31 23:42:49.670000+00:00  38.790833 -122.760834  0.85   
2  us7000fqca  2021-10-31 23:39:39.089000+00:00 -25.318900  179.162600  4.40   
3  hv72777307  2021-10-31 23:37:23.860000+00:00  19.201667 -155.412833  1.52   
4  ci39845863  2021-10-31 23:34:04.820000+00:00  35.711167 -117.583833  1.46   

                        place  
0    235 km E of Levuka, Fiji  
1  2km NNW of The Geysers, CA  
2   south of the Fiji Islands  
3    6 km E of P?hala, Hawaii  
4   13km NE of Ridgecrest, CA  


In [6]:
earthquake_pairs_df = df[
    ["id", "time", "latitude", "longitude", "mag", "place"]
].copy()

earthquake_pairs_df["time"] = pd.to_datetime(
    earthquake_pairs_df["time"],
    format="mixed",
    utc=True,
    errors="coerce"
)

earthquake_pairs_df = earthquake_pairs_df.dropna(
    subset=["id", "time", "latitude", "longitude"]
)

earthquake_pairs_df = earthquake_pairs_df[
    earthquake_pairs_df["latitude"].between(-90, 90)
    & earthquake_pairs_df["longitude"].between(-180, 180)
].reset_index(drop=True)

print("Prepared records:", len(earthquake_pairs_df))

Prepared records: 739785


In [ ]:
# Convert coordinates to radians
coordinates_rad = np.radians(
    earthquake_pairs_df[["latitude", "longitude"]].to_numpy()
)

# Build the spatial index
earthquake_tree = BallTree(coordinates_rad, metric="haversine")

# Convert 50 km into radians
earth_radius_km = 6371.0088
radius_rad = 50 / earth_radius_km

# Search for nearby events
nearby_indices = earthquake_tree.query_radius(
    coordinates_rad,
    r=radius_rad,
    return_distance=False
)

# Count nearby relationships, excluding each event itself
candidate_count = sum(len(indices) - 1 for indices in nearby_indices)

print("Nearby candidate relationships:", candidate_count)
print("Spatial search completed successfully.")

In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

# Reload the dataset after the kernel restart
df = pd.read_csv(
    "/Users/shilpathamaraiselvan/earthquakes_5years_processed.csv"
)

earthquake_pairs_df = df[
    ["id", "time", "latitude", "longitude", "mag", "place"]
].copy()

earthquake_pairs_df["time"] = pd.to_datetime(
    earthquake_pairs_df["time"],
    format="mixed",
    utc=True,
    errors="coerce"
)

earthquake_pairs_df = earthquake_pairs_df.dropna(
    subset=["id", "time", "latitude", "longitude"]
)

earthquake_pairs_df = earthquake_pairs_df[
    earthquake_pairs_df["latitude"].between(-90, 90)
    & earthquake_pairs_df["longitude"].between(-180, 180)
].reset_index(drop=True)

# Build the spatial index
coordinates_rad = np.radians(
    earthquake_pairs_df[["latitude", "longitude"]].to_numpy()
)
earthquake_tree = BallTree(coordinates_rad, metric="haversine")

# Convert timestamps to seconds for fast comparisons
time_seconds = (
    earthquake_pairs_df["time"].astype("int64").to_numpy() / 1e9
)

earth_radius_km = 6371.0088
radius_rad = 50 / earth_radius_km
one_hour_seconds = 3600
batch_size = 5000

matched_pairs = []

for start in range(0, len(earthquake_pairs_df), batch_size):
    end = min(start + batch_size, len(earthquake_pairs_df))

    # Query only this batch; don't retain all neighbour lists
    nearby_indices = earthquake_tree.query_radius(
        coordinates_rad[start:end],
        r=radius_rad,
        return_distance=False
    )

    for offset, neighbors in enumerate(nearby_indices):
        i = start + offset

        # Keep each pair only once
        neighbors = neighbors[neighbors > i]

        if len(neighbors) == 0:
            continue

        # Keep pairs within one hour
        close_in_time = (
            np.abs(time_seconds[neighbors] - time_seconds[i])
            <= one_hour_seconds
        )
        neighbors = neighbors[close_in_time]

        for j in neighbors:
            matched_pairs.append((i, int(j)))

    if start % 50000 == 0:
        print(f"Processed {end:,} of {len(earthquake_pairs_df):,} records")

print("Matching pairs found:", len(matched_pairs))

Processed 5,000 of 739,785 records
Processed 55,000 of 739,785 records
Processed 105,000 of 739,785 records
Processed 155,000 of 739,785 records
Processed 205,000 of 739,785 records
Processed 255,000 of 739,785 records
Processed 305,000 of 739,785 records
Processed 355,000 of 739,785 records
Processed 405,000 of 739,785 records
Processed 455,000 of 739,785 records
Processed 505,000 of 739,785 records
Processed 555,000 of 739,785 records
Processed 605,000 of 739,785 records
Processed 655,000 of 739,785 records
Processed 705,000 of 739,785 records
Matching pairs found: 1011290


In [3]:
# Convert matching index pairs into a DataFrame
pair_rows = []

for i, j in matched_pairs:
    event_a = earthquake_pairs_df.iloc[i]
    event_b = earthquake_pairs_df.iloc[j]

    # Calculate the exact great-circle distance
    distance_rad = np.arccos(
        np.clip(
            np.sin(np.radians(event_a["latitude"]))
            * np.sin(np.radians(event_b["latitude"]))
            + np.cos(np.radians(event_a["latitude"]))
            * np.cos(np.radians(event_b["latitude"]))
            * np.cos(
                np.radians(event_a["longitude"] - event_b["longitude"])
            ),
            -1,
            1
        )
    )

    distance_km = 6371.0088 * distance_rad

    pair_rows.append({
        "event_id_1": event_a["id"],
        "time_1": event_a["time"],
        "place_1": event_a["place"],
        "magnitude_1": event_a["mag"],
        "event_id_2": event_b["id"],
        "time_2": event_b["time"],
        "place_2": event_b["place"],
        "magnitude_2": event_b["mag"],
        "distance_km": distance_km,
        "time_difference_minutes": abs(
            (event_b["time"] - event_a["time"]).total_seconds()
        ) / 60
    })

earthquake_pair_results = pd.DataFrame(pair_rows)

# Sort by closest distance
earthquake_pair_results = earthquake_pair_results.sort_values(
    ["distance_km", "time_difference_minutes"]
).reset_index(drop=True)

print("Total matching pairs:", len(earthquake_pair_results))
display(earthquake_pair_results.head(10))

Total matching pairs: 1011290


,event_id_1,time_1,place_1,magnitude_1,event_id_2,time_2,place_2,magnitude_2,distance_km,time_difference_minutes
0,ak021dz4k1sj,2021-10-31 22:14:04.352000+00:00,"9 km NNW of Big Lake, Alaska",1.6,ak021dz4k1qq,2021-10-31 22:14:04.352000+00:00,"9 km NNW of Big Lake, Alaska",1.6,0.0,0.0
1,ak021dyynjq6,2021-10-31 12:22:59.502000+00:00,"27 km SW of Atka, Alaska",1.6,ak021dyynmw5,2021-10-31 12:22:59.502000+00:00,"27 km SW of Atka, Alaska",1.6,0.0,0.0
2,ak021dyv04yv,2021-10-31 06:09:52.956000+00:00,"63 km S of Glacier View, Alaska",1.4,ak021dyv05a7,2021-10-31 06:09:52.956000+00:00,"63 km S of Glacier View, Alaska",1.4,0.0,0.0
3,ak021dxandpo,2021-10-30 11:44:09.781000+00:00,"Rat Islands, Aleutian Islands, Alaska",1.9,ak021dxandjm,2021-10-30 11:44:09.781000+00:00,"Rat Islands, Aleutian Islands, Alaska",1.9,0.0,0.0
4,ak021dx5nyx5,2021-10-30 03:07:06.019000+00:00,"6 km SSW of Halibut Cove, Alaska",1.4,ak021dx5nyog,2021-10-30 03:07:06.019000+00:00,"6 km SSW of Halibut Cove, Alaska",1.4,0.0,0.0
5,ak021dvkso6j,2021-10-29 07:51:34.931000+00:00,"7 km SW of Anchorage, Alaska",1.1,ak021dvkruiv,2021-10-29 07:51:34.931000+00:00,"7 km SW of Anchorage, Alaska",1.1,0.0,0.0
6,ak021ds5ixiy,2021-10-27 00:50:46.342000+00:00,"63 km NW of Tatitlek, Alaska",1.1,ak021ds5j1sk,2021-10-27 00:50:46.342000+00:00,"63 km NW of Tatitlek, Alaska",1.1,0.0,0.0
7,ak021dp80prt,2021-10-25 23:29:50.974000+00:00,"128 km W of Elfin Cove, Alaska",2.5,ak021dp80pxg,2021-10-25 23:29:50.974000+00:00,"128 km W of Elfin Cove, Alaska",2.5,0.0,0.0
8,ak021dp802h8,2021-10-25 23:26:49.590000+00:00,"13 km WSW of Dutch Harbor, Alaska",1.5,ak021dp802in,2021-10-25 23:26:49.590000+00:00,"13 km WSW of Dutch Harbor, Alaska",1.5,0.0,0.0
9,ak021dp6rte7,2021-10-25 21:20:31.125000+00:00,"107 km E of Cordova, Alaska",0.9,ak021dp6rvl4,2021-10-25 21:20:31.125000+00:00,"107 km E of Cordova, Alaska",0.9,0.0,0.0


In [4]:
# Check how many pairs have the same event ID
same_id_count = sum(
    earthquake_pairs_df.iloc[i]["id"]
    == earthquake_pairs_df.iloc[j]["id"]
    for i, j in matched_pairs
)

print("Pairs with identical event IDs:", same_id_count)
print("Total candidate pairs:", len(matched_pairs))

Pairs with identical event IDs: 0
Total candidate pairs: 1011290


In [5]:
# Build a DataFrame of matched pairs
pair_indices = np.array(matched_pairs, dtype=np.int32)

i = pair_indices[:, 0]
j = pair_indices[:, 1]

lat1 = coordinates_rad[i, 0]
lon1 = coordinates_rad[i, 1]
lat2 = coordinates_rad[j, 0]
lon2 = coordinates_rad[j, 1]

# Haversine distance in kilometres
dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
)

distance_km = 2 * earth_radius_km * np.arcsin(
    np.sqrt(np.clip(a, 0, 1))
)

time_diff_minutes = (
    np.abs(time_seconds[j] - time_seconds[i]) / 60
)

# Validate both conditions
valid = (distance_km <= 50) & (time_diff_minutes <= 60)

print("Candidate pairs:", len(matched_pairs))
print("Pairs passing distance and time checks:", int(valid.sum()))
print("Pairs failing validation:", int((~valid).sum()))

Candidate pairs: 1011290
Pairs passing distance and time checks: 1011290
Pairs failing validation: 0


In [6]:
# Keep only validated pairs
valid_indices = np.where(valid)[0]

i_valid = i[valid_indices]
j_valid = j[valid_indices]

earthquake_pair_results = pd.DataFrame({
    "event_id_1": earthquake_pairs_df.iloc[i_valid]["id"].to_numpy(),
    "time_1": earthquake_pairs_df.iloc[i_valid]["time"].to_numpy(),
    "place_1": earthquake_pairs_df.iloc[i_valid]["place"].to_numpy(),
    "magnitude_1": earthquake_pairs_df.iloc[i_valid]["mag"].to_numpy(),
    "event_id_2": earthquake_pairs_df.iloc[j_valid]["id"].to_numpy(),
    "time_2": earthquake_pairs_df.iloc[j_valid]["time"].to_numpy(),
    "place_2": earthquake_pairs_df.iloc[j_valid]["place"].to_numpy(),
    "magnitude_2": earthquake_pairs_df.iloc[j_valid]["mag"].to_numpy(),
    "distance_km": distance_km[valid_indices],
    "time_difference_minutes": time_diff_minutes[valid_indices]
})

# Save results
output_file = (
    "/Users/shilpathamaraiselvan/"
    "earthquake_pairs_within_50km_1hour.csv"
)

earthquake_pair_results.to_csv(output_file, index=False)

print("Validated pairs:", len(earthquake_pair_results))
print("Saved file:", output_file)
display(earthquake_pair_results.head(10))

Validated pairs: 1011290
Saved file: /Users/shilpathamaraiselvan/earthquake_pairs_within_50km_1hour.csv


,event_id_1,time_1,place_1,magnitude_1,event_id_2,time_2,place_2,magnitude_2,distance_km,time_difference_minutes
0,nc73647215,2021-10-31 23:42:49.670000+00:00,"2km NNW of The Geysers, CA",0.85,nc73647240,2021-11-01 00:42:19.920000+00:00,"8km NW of The Geysers, CA",0.80,6.217554,59.504167
1,hv72777307,2021-10-31 23:37:23.860000+00:00,"6 km E of P?hala, Hawaii",1.52,hv72777282,2021-10-31 23:19:18.820000+00:00,"5 km S of P?hala, Hawaii",1.67,9.384924,18.084000
2,hv72777307,2021-10-31 23:37:23.860000+00:00,"6 km E of P?hala, Hawaii",1.52,hv72777367,2021-11-01 00:34:58.810000+00:00,"Island of Hawaii, Hawaii",1.87,8.739560,57.582500
3,hv72777307,2021-10-31 23:37:23.860000+00:00,"6 km E of P?hala, Hawaii",1.52,hv72777247,2021-10-31 22:51:28.950000+00:00,"4 km S of P?hala, Hawaii",2.02,8.116439,45.915167
4,hv72777307,2021-10-31 23:37:23.860000+00:00,"6 km E of P?hala, Hawaii",1.52,hv72777352,2021-11-01 00:18:25.960000+00:00,"2 km S of P?hala, Hawaii",2.19,7.335809,41.035000
5,hv72777307,2021-10-31 23:37:23.860000+00:00,"6 km E of P?hala, Hawaii",1.52,hv72777287,2021-10-31 23:24:00.470000+00:00,"5 km SSE of P?hala, Hawaii",1.85,6.280261,13.389833
6,hv72777307,2021-10-31 23:37:23.860000+00:00,"6 km E of P?hala, Hawaii",1.52,hv72000303,2021-11-01 00:13:16.690000+00:00,"8 km E of P?hala, Hawaii",1.97,1.191980,35.880500
7,hv72777307,2021-10-31 23:37:23.860000+00:00,"6 km E of P?hala, Hawaii",1.52,hv72777277,2021-10-31 23:12:16.450000+00:00,"7 km E of P?hala, Hawaii",3.06,1.497401,25.123500
8,hv72777307,2021-10-31 23:37:23.860000+00:00,"6 km E of P?hala, Hawaii",1.52,hv72777332,2021-11-01 00:13:19.780000+00:00,"8 km E of P?hala, Hawaii",2.27,2.102219,35.932000
9,ci39845863,2021-10-31 23:34:04.820000+00:00,"13km NE of Ridgecrest, CA",1.46,ci39845839,2021-10-31 22:47:28.170000+00:00,"13km WSW of Searles Valley, CA",1.01,4.240738,46.610833


In [7]:
output_file = (
    "/Users/shilpathamaraiselvan/"
    "earthquake_pairs_within_50km_1hour.csv"
)

earthquake_pair_results.to_csv(output_file, index=False)

print("Question 29 completed successfully.")
print("Total matching pairs:", len(earthquake_pair_results))
print("CSV saved to:", output_file)

Question 29 completed successfully.
Total matching pairs: 1011290
CSV saved to: /Users/shilpathamaraiselvan/earthquake_pairs_within_50km_1hour.csv


In [8]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        location_region,
        COUNT(*) AS deep_earthquake_count,
        ROUND(AVG(depth_km), 2) AS average_depth_km,
        ROUND(MAX(depth_km), 2) AS maximum_depth_km
    FROM earthquakes
    WHERE depth_km > 300
      AND location_region IS NOT NULL
      AND location_region <> ''
    GROUP BY location_region
    ORDER BY deep_earthquake_count DESC
    LIMIT 10;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

NameError: name 'conn' is not defined

In [9]:
import mysql.connector
from getpass import getpass

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=getpass("Enter your MySQL password: "),
    database="global_seismic_trends"
)

print("MySQL connected successfully:", conn.is_connected())

Enter your MySQL password:  ········


MySQL connected successfully: True


In [10]:
cursor = conn.cursor()

cursor.execute("""
    SELECT
        location_region,
        COUNT(*) AS deep_earthquake_count,
        ROUND(AVG(depth_km), 2) AS average_depth_km,
        ROUND(MAX(depth_km), 2) AS maximum_depth_km
    FROM earthquakes
    WHERE depth_km > 300
      AND location_region IS NOT NULL
      AND location_region <> ''
    GROUP BY location_region
    ORDER BY deep_earthquake_count DESC
    LIMIT 10;
""")

for row in cursor.fetchall():
    print(row)

cursor.close()

('south of the Fiji Islands', 1511, 522.6, 660.0)
('Fiji region', 1127, 558.27, 671.04)
('Fiji', 1071, 566.13, 669.56)
('Tonga', 548, 473.51, 623.32)
('Indonesia', 252, 456.81, 664.57)
('Japan region', 220, 447.5, 580.25)
('Timor Leste', 191, 445.19, 605.86)
('Kermadec Islands region', 150, 394.59, 587.55)
('Wallis and Futuna', 138, 404.17, 615.53)
('Japan', 119, 363.06, 501.6)


True

In [11]:
import importlib.util

packages = ["streamlit", "mysql.connector", "pandas", "plotly"]

for package in packages:
    module_name = package.split(".")[0]
    installed = importlib.util.find_spec(package) is not None
    print(f"{package}: {'Installed' if installed else 'Not installed'}")

streamlit: Installed
mysql.connector: Installed
pandas: Installed
plotly: Installed


In [12]:
import mysql.connector
from getpass import getpass

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=getpass("Enter your MySQL password: "),
    database="global_seismic_trends"
)

cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM earthquakes")
print("Total MySQL records:", cursor.fetchone()[0])

cursor.execute("""
    SELECT COUNT(*)
    FROM earthquakes
    WHERE mag IS NOT NULL
      AND depth_km IS NOT NULL
      AND year IS NOT NULL
""")
print("Records with magnitude, depth and year:", cursor.fetchone()[0])

cursor.close()
conn.close()

Enter your MySQL password:  ········


Total MySQL records: 739785
Records with magnitude, depth and year: 739730
